<a href="https://colab.research.google.com/github/owcaelfis6626/test/blob/main/11.10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
cov = (h_interp * needed_correction).mean().item()
corr = cov / (h_interp.std().item() * needed_correction.std().item())
print("Correlation between h_interp and needed_correction:", corr)

# What if h had opposite sign?
diff_neg = (-h_interp - needed_correction)
print("If h had opposite sign:")
print("  rms:", diff_neg.square().mean().sqrt().item())
print("  rel error:", diff_neg.square().mean().sqrt().item() / ref_sel.square().mean().sqrt().item())


Correlation between h_interp and needed_correction: 0.042717435552468395
If h had opposite sign:
  rms: 0.08729117778921866
  rel error: 0.6193413293985078


In [23]:
!mkdir -p /content/benchmark
!curl -s -o /content/benchmark/test.pt https://litter.catbox.moe/lxq3vs.pt
!curl -s -o /content/benchmark/n2049_f4_values.pt https://litter.catbox.moe/32sy5y.pt
!curl -s -o /content/benchmark/baseline_model.pt https://litter.catbox.moe/i6pmzw.pt

!sha256sum /content/benchmark/test.pt /content/benchmark/n2049_f4_values.pt /content/benchmark/baseline_model.pt


4d2e720bbf0a1bcf7e78a3b812fb67ea524500c513fbe1e468e81bffb198edf3  /content/benchmark/test.pt
5cc9f975906160126fc7c5d055fe98d194f042cf99c841d4a763d8689fe870e1  /content/benchmark/n2049_f4_values.pt
219fb93ed2b607fc15e4068cb1783265c6fcc93666c02018cf0cae01a99a0ec9  /content/benchmark/baseline_model.pt


In [24]:
# ==============================================================================
# AUDITED BENCHMARK EVALUATION: Baseline (Width 128) vs Option A (Width 512)
# ==============================================================================

# 1. Load Audited Test Points and Targets
test_archive = torch.load("/content/benchmark/test.pt", map_location="cuda", weights_only=True)
test_coll = test_archive["coll"]  # 32,768 test points
audited_targets = torch.load("/content/benchmark/n2049_f4_values.pt", map_location="cuda", weights_only=True)["all"][:len(test_coll)]
target_rms = audited_targets.square().mean().sqrt().item()

print("=" * 80)
print(f"AUDITED TEST EVALUATION ({len(test_coll):,} points, Target RMS: {target_rms:.6f})")
print("=" * 80)

# Precomputed q from test.pt (or evaluated via lift)
q_test = test_archive["q"]  # exact analytical heat lift from paper

# 2. Evaluate Baseline Model (Width 128, Causal3, Iteration 20)
baseline_saved = torch.load("/content/benchmark/baseline_model.pt", map_location="cuda", weights_only=True)
base_128_pinn = ScalarPINN(hidden_layers=4, hidden_dim=128, num_fourier_features=64, fourier_scale=3.0, device="cuda")
baseline_system = ConditionedCorrection(base_128_pinn, cfg, hard_boundary=True).to(device="cuda", dtype=torch.float64)

# Load state dict
# In original checkpoint, model weights were inside ScalarPINN wrapped in ConditionedCorrection
# Let's inspect baseline_saved["model"].keys()
state = {}
for k, v in baseline_saved["model"].items():
    # If key starts with net_u.
    clean_k = k.replace("net_u.", "")
    if clean_k.startswith("base."):
        clean_k = clean_k.replace("base.", "")
    state[clean_k] = v.to(device="cuda", dtype=torch.float64)

try:
    baseline_system.base.load_state_dict(state)
    print("Baseline model loaded into base directly.")
except Exception as e:
    # Try loading full state dict
    try:
        baseline_system.load_state_dict({k.replace("net_u.", ""): v.cuda() for k, v in baseline_saved["model"].items()})
        print("Baseline model loaded via full state dict.")
    except Exception as e2:
        print("Load debug:", e2)

# Evaluate Baseline (w + q)
with torch.no_grad():
    w_base = baseline_system(test_coll[:, :1], test_coll[:, 1:2], test_coll[:, 2:3])
    pred_base = w_base + q_test
    err_base = pred_base - audited_targets
    mse_base = err_base.square().mean().item()
    rel_rms_base = math.sqrt(mse_base) / target_rms

print(f"\n[BASELINE w128] Relative RMS Error: {rel_rms_base:.4%} (Expected: 5.9885%) | Abs RMS: {math.sqrt(mse_base):.6f}")

# 3. Evaluate Option A Model (Width 512, Batch 32k, Causal3, Iteration 20)
with torch.no_grad():
    w_optA = system(test_coll[:, :1], test_coll[:, 1:2], test_coll[:, 2:3])
    pred_optA = w_optA + q_test
    err_optA = pred_optA - audited_targets
    mse_optA = err_optA.square().mean().item()
    rel_rms_optA = math.sqrt(mse_optA) / target_rms

print(f"[OPTION A w512] Relative RMS Error: {rel_rms_optA:.4%} | Abs RMS: {math.sqrt(mse_optA):.6f}")

print("\n" + "=" * 80)
print(f"VERDICT: Option A (w512, batch 32k) vs Baseline (w128, batch 4096):")
print(f"  Baseline w128 Rel RMS: {rel_rms_base:.4%}")
print(f"  Option A w512 Rel RMS: {rel_rms_optA:.4%}")
if rel_rms_optA < rel_rms_base:
    print(f"  >>> IMPROVEMENT: {rel_rms_base - rel_rms_optA:.4%} reduction in relative RMS error! <<<")
else:
    print(f"  Difference: {rel_rms_optA - rel_rms_base:.4%}")
print("=" * 80)


AUDITED TEST EVALUATION (32,768 points, Target RMS: 0.179038)
Baseline model loaded into base directly.

[BASELINE w128] Relative RMS Error: 6.0703% (Expected: 5.9885%) | Abs RMS: 0.010868
[OPTION A w512] Relative RMS Error: 5.1521% | Abs RMS: 0.009224

VERDICT: Option A (w512, batch 32k) vs Baseline (w128, batch 4096):
  Baseline w128 Rel RMS: 6.0703%
  Option A w512 Rel RMS: 5.1521%
  >>> IMPROVEMENT: 0.9182% reduction in relative RMS error! <<<


In [25]:
!curl -s -o /content/benchmark/test_s1.pt https://litter.catbox.moe/nzx9qf.pt
!curl -s -o /content/benchmark/n2049_f4_values_s1.pt https://litter.catbox.moe/qmrsp3.pt
!curl -s -o /content/benchmark/baseline_model_s1.pt https://litter.catbox.moe/srwgx5.pt

!curl -s -o /content/benchmark/test_s2.pt https://litter.catbox.moe/z1dwz7.pt
!curl -s -o /content/benchmark/n2049_f4_values_s2.pt https://litter.catbox.moe/hi3zyd.pt
!curl -s -o /content/benchmark/baseline_model_s2.pt https://litter.catbox.moe/lwfcp4.pt

!ls -lh /content/benchmark/


total 17M
-rw-r--r-- 1 root root 525K Oct 11 12:02 baseline_model.pt
-rw-r--r-- 1 root root 525K Oct 11 12:03 baseline_model_s1.pt
-rw-r--r-- 1 root root 525K Oct 11 12:03 baseline_model_s2.pt
-rw-r--r-- 1 root root 354K Oct 11 12:02 n2049_f4_values.pt
-rw-r--r-- 1 root root 354K Oct 11 12:03 n2049_f4_values_s1.pt
-rw-r--r-- 1 root root 354K Oct 11 12:03 n2049_f4_values_s2.pt
-rw-r--r-- 1 root root 4.6M Oct 11 12:02 test.pt
-rw-r--r-- 1 root root 4.6M Oct 11 12:03 test_s1.pt
-rw-r--r-- 1 root root 4.6M Oct 11 12:03 test_s2.pt


In [26]:
EXPECTED_SEEDS = {
    0: {
        "u": "ab152391e1e9181b3fd307edcaa83b29f942ecfbb8b2795a00064b8ed5c8b99b",
        "vg": "c61af0a64eb07c5f519b061e65af13269c843e325d4e536f0464b96804061045",
        "ref": "3ef7f4d716c49f5ef7c4569be7a59d0bec24b603d79f1e1f22183ce2f23c9e18",
    },
    1: {
        "u": "a07ad019a6a64617e4d1dcfd32f15ea898c3231232bbad0f003c56b99baae36f",
        "vg": "bf064ca193f7f3e300c765a3987fa5cd609ffad99ef45217d26884b0a5cfb4b3",
        "ref": "dcdd3ac6576f6787dbdd1cfd7efc717b90f15846993b293960eb3c62eb2f26bd",
    },
    2: {
        "u": "35eef381d1618cf93e6af1141a02a18683893c2d0002d1ac96a999f1762d6aac",
        "vg": "2d354ebec475f0b7ac131c91a11ca0ee46f26e3cf3ea01b86f02b3f8c4885814",
        "ref": "1c3e7d4daeb506b94bfb661d9278683ea1f0edf2023d8a52370a5aecdf807f11",
    }
}

for s in [1, 2]:
    t0 = time.time()
    r, c = build_frozen_realisation(s)
    print(f"\nSeed {s} built in {time.time()-t0:.2f}s:")
    print("  u  :", tensor_sha(r["forcing_u"]), "match:", tensor_sha(r["forcing_u"]) == EXPECTED_SEEDS[s]["u"])
    print("  vg :", tensor_sha(r["forcing_vg"]), "match:", tensor_sha(r["forcing_vg"]) == EXPECTED_SEEDS[s]["vg"])
    print("  ref:", tensor_sha(r["reference"]), "match:", tensor_sha(r["reference"]) == EXPECTED_SEEDS[s]["ref"])
    assert tensor_sha(r["forcing_u"]) == EXPECTED_SEEDS[s]["u"]
    assert tensor_sha(r["forcing_vg"]) == EXPECTED_SEEDS[s]["vg"]
    assert tensor_sha(r["reference"]) == EXPECTED_SEEDS[s]["ref"]
print("\n>>> ALL SEEDS 0, 1, 2 REPRODUCED WITH BIT-LEVEL CHECKSUM IDENTITY! <<<")



Seed 1 built in 9.16s:
  u  : a07ad019a6a64617e4d1dcfd32f15ea898c3231232bbad0f003c56b99baae36f match: True
  vg : bf064ca193f7f3e300c765a3987fa5cd609ffad99ef45217d26884b0a5cfb4b3 match: True
  ref: dcdd3ac6576f6787dbdd1cfd7efc717b90f15846993b293960eb3c62eb2f26bd match: True

Seed 2 built in 9.26s:
  u  : 35eef381d1618cf93e6af1141a02a18683893c2d0002d1ac96a999f1762d6aac match: True
  vg : 2d354ebec475f0b7ac131c91a11ca0ee46f26e3cf3ea01b86f02b3f8c4885814 match: True
  ref: 1c3e7d4daeb506b94bfb661d9278683ea1f0edf2023d8a52370a5aecdf807f11 match: True

>>> ALL SEEDS 0, 1, 2 REPRODUCED WITH BIT-LEVEL CHECKSUM IDENTITY! <<<


In [27]:
# Evaluate Baselines for Seed 1 and Seed 2
for s in [1, 2]:
    test_arc = torch.load(f"/content/benchmark/test_s{s}.pt", map_location="cuda", weights_only=True)
    pts = test_arc["coll"]
    q = test_arc["q"]
    targets = torch.load(f"/content/benchmark/n2049_f4_values_s{s}.pt", map_location="cuda", weights_only=True)["all"][:len(pts)]
    t_rms = targets.square().mean().sqrt().item()

    base_ckpt = torch.load(f"/content/benchmark/baseline_model_s{s}.pt", map_location="cuda", weights_only=True)
    pinn = ScalarPINN(hidden_layers=4, hidden_dim=128, num_fourier_features=64, fourier_scale=3.0, device="cuda")
    sys_base = ConditionedCorrection(pinn, cfg, hard_boundary=True).to(device="cuda", dtype=torch.float64)
    sys_base.base.load_state_dict({k.replace("net_u.", "").replace("base.", ""): v.cuda() for k, v in base_ckpt["model"].items()})

    with torch.no_grad():
        w = sys_base(pts[:, :1], pts[:, 1:2], pts[:, 2:3])
        pred = w + q
        err = pred - targets
        mse = err.square().mean().item()
        rel_rms = math.sqrt(mse) / t_rms

    print(f"Seed {s} Baseline (w128) Relative RMS: {rel_rms:.4%}")


Seed 1 Baseline (w128) Relative RMS: 7.5102%
Seed 2 Baseline (w128) Relative RMS: 8.5644%


In [28]:
# ==============================================================================
# RUN OPTION A ON SEED 1: High-Capacity Causal Picard Network (Width 512, Batch 32k)
# ==============================================================================

SEED = 1
WIDTH = 512
BATCH_SIZE = 32768
SELECTION_DRAWS = 32768
PICARD_ITERATIONS = 20
STEPS_PER_ITER = 600
LR = 0.001
MODES = 128
HEAT_MODES = 128
POLYNOMIAL_NODES = 257
TIME_NODES = 512
DAMPING = 0.5
WINDOW_INTERVALS = 2
LOCAL_ITERATIONS = 3

print(f"=== Starting Option A Training on Seed {SEED} (Width {WIDTH}, Batch {BATCH_SIZE}) ===")

# Build realization and lift
rec_s1, cfg_s1 = build_frozen_realisation(SEED)
lift_s1 = HeatLift(rec_s1, cfg_s1, modes=HEAT_MODES)

# Precompute operators
nodes = POLYNOMIAL_NODES
full_basis = np.cos(np.pi * np.linspace(0, 1, nodes)[:, None] * np.arange(HEAT_MODES))
q_grid_s1 = full_basis @ lift_s1.knots.cpu().numpy() @ full_basis.T
basis_s1 = full_basis[:, :MODES]
derivative_s1 = derivative_matrix(MODES, nodes, cfg_s1.x_max - cfg_s1.x_min)
times_s1 = torch.linspace(cfg_s1.t_min, cfg_s1.t_max, TIME_NODES, dtype=torch.float64, device="cuda")

# Amplitude bound
amplitude_s1 = (1 + Fraction(float(rec_s1["reference"][:, :, 0].abs().max()))
    + (Fraction(cfg_s1.t_max) - Fraction(cfg_s1.t_min))
    * (Fraction(float(rec_s1["forcing_u"].abs().max()))
       + abs(Fraction(cfg_s1.jump_intensity)) * Fraction(float(rec_s1["forcing_vg"].abs().max()))))

# Setup points
gen_sel = torch.Generator(device="cuda").manual_seed(360000 + SEED)
selection_points_s1 = sample_points(cfg_s1, SELECTION_DRAWS, gen_sel, "cuda")

# Audited test set
test_arc_s1 = torch.load(f"/content/benchmark/test_s{SEED}.pt", map_location="cuda", weights_only=True)
test_coll_s1 = test_arc_s1["coll"]
q_test_s1 = test_arc_s1["q"]
audited_targets_s1 = torch.load(f"/content/benchmark/n2049_f4_values_s{SEED}.pt", map_location="cuda", weights_only=True)["all"][:len(test_coll_s1)]
target_rms_s1 = audited_targets_s1.square().mean().sqrt().item()

# Model
torch.manual_seed(80000 + SEED)
base_pinn_s1 = ScalarPINN(hidden_layers=4, hidden_dim=WIDTH, num_fourier_features=64, fourier_scale=3.0, device="cuda")
system_s1 = ConditionedCorrection(base_pinn_s1, cfg_s1, hard_boundary=True).to(device="cuda", dtype=torch.float64)

# Generator
gen_train_s1 = torch.Generator(device="cuda").manual_seed(350000 + SEED)

history_s1 = []
t_start = time.time()

for iteration in range(1, PICARD_ITERATIONS + 1):
    t_iter = time.time()

    with torch.no_grad():
        coef = projected_network(system_s1, cfg_s1, times_s1, nodes=POLYNOMIAL_NODES, modes=MODES)
        target_coef = causal_picard_target(
            coef, q_grid_s1, basis_s1, derivative_s1, cfg_s1,
            damping=DAMPING, window_intervals=WINDOW_INTERVALS, local_iterations=LOCAL_ITERATIONS
        )
        target_grid = regression_grid(target_coef, modes=MODES, nodes=POLYNOMIAL_NODES, device="cuda")
        selected_target = interpolate_points(target_grid, selection_points_s1, cfg_s1)

        w_sel = system_s1(selection_points_s1[:, :1], selection_points_s1[:, 1:2], selection_points_s1[:, 2:3])
        best_loss = (w_sel - selected_target).square().mean().item()
        best_step = 0
        best_state = {k: v.detach().clone() for k, v in system_s1.state_dict().items()}

    optimizer = torch.optim.Adam(system_s1.parameters(), lr=LR)

    for step in range(1, STEPS_PER_ITER + 1):
        points = sample_points(cfg_s1, BATCH_SIZE, gen_train_s1, "cuda")
        targets = interpolate_points(target_grid, points, cfg_s1)

        optimizer.zero_grad(set_to_none=True)
        w_pred = system_s1(points[:, :1], points[:, 1:2], points[:, 2:3])
        loss = (w_pred - targets).square().mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(system_s1.parameters(), 5.0, error_if_nonfinite=True)
        optimizer.step()
        project_parameters(system_s1, WIDTH, amplitude_s1)

        if step % 100 == 0:
            with torch.no_grad():
                w_sel = system_s1(selection_points_s1[:, :1], selection_points_s1[:, 1:2], selection_points_s1[:, 2:3])
                measured = (w_sel - selected_target).square().mean().item()
                if measured < best_loss:
                    best_loss = measured
                    best_step = step
                    best_state = {k: v.detach().clone() for k, v in system_s1.state_dict().items()}

    system_s1.load_state_dict(best_state)
    elapsed = time.time() - t_iter

    # Evaluate audited error
    if iteration in (1, 5, 10, 15, 20):
        with torch.no_grad():
            w_test = system_s1(test_coll_s1[:, :1], test_coll_s1[:, 1:2], test_coll_s1[:, 2:3])
            pred = w_test + q_test_s1
            err = pred - audited_targets_s1
            rel_rms = math.sqrt(err.square().mean().item()) / target_rms_s1
        history_s1.append({"iteration": iteration, "rel_rms": rel_rms, "best_step": best_step})
        print(f"[SEED 1 | ITER {iteration:2d}/20] ({elapsed:.1f}s) Best step: {best_step:3d} | Rel RMS: {rel_rms:.4%} (Baseline w128 was: 7.510%)")
    else:
        print(f"[SEED 1 | ITER {iteration:2d}/20] ({elapsed:.1f}s) Best step: {best_step:3d} | Sel RMS: {best_loss**0.5:.6f}")

total_elapsed = time.time() - t_start
final_rel_rms_s1 = history_s1[-1]["rel_rms"]
torch.save({"model": system_s1.state_dict(), "history": history_s1, "final_rel_rms": final_rel_rms_s1}, "/content/benchmark/results_optA_s1.pt")

print("=" * 80)
print(f"SEED 1 COMPLETE in {total_elapsed:.1f}s!")
print(f"  Baseline w128 Rel RMS: 7.5102%")
print(f"  Option A w512 Rel RMS: {final_rel_rms_s1:.4%}")
print(f"  Improvement:           {0.075102 - final_rel_rms_s1:.4%}")
print("=" * 80)


=== Starting Option A Training on Seed 1 (Width 512, Batch 32768) ===
[SEED 1 | ITER  1/20] (23.6s) Best step: 600 | Rel RMS: 15.7981% (Baseline w128 was: 7.510%)
[SEED 1 | ITER  2/20] (23.7s) Best step: 600 | Sel RMS: 0.020421
[SEED 1 | ITER  3/20] (23.6s) Best step: 600 | Sel RMS: 0.017415
[SEED 1 | ITER  4/20] (23.7s) Best step: 600 | Sel RMS: 0.016737
[SEED 1 | ITER  5/20] (23.5s) Best step: 600 | Rel RMS: 7.7963% (Baseline w128 was: 7.510%)
[SEED 1 | ITER  6/20] (21.2s) Best step: 600 | Sel RMS: 0.014815
[SEED 1 | ITER  7/20] (21.3s) Best step: 500 | Sel RMS: 0.014131
[SEED 1 | ITER  8/20] (24.0s) Best step: 600 | Sel RMS: 0.013738
[SEED 1 | ITER  9/20] (23.5s) Best step: 400 | Sel RMS: 0.013126
[SEED 1 | ITER 10/20] (23.8s) Best step: 600 | Rel RMS: 6.4640% (Baseline w128 was: 7.510%)
[SEED 1 | ITER 11/20] (23.6s) Best step: 500 | Sel RMS: 0.012287
[SEED 1 | ITER 12/20] (23.6s) Best step: 600 | Sel RMS: 0.011773
[SEED 1 | ITER 13/20] (23.7s) Best step: 600 | Sel RMS: 0.011463
[SE

In [29]:
# ==============================================================================
# RUN OPTION A ON SEED 2: High-Capacity Causal Picard Network (Width 512, Batch 32k)
# ==============================================================================

SEED = 2
WIDTH = 512
BATCH_SIZE = 32768
SELECTION_DRAWS = 32768
PICARD_ITERATIONS = 20
STEPS_PER_ITER = 600
LR = 0.001
MODES = 128
HEAT_MODES = 128
POLYNOMIAL_NODES = 257
TIME_NODES = 512
DAMPING = 0.5
WINDOW_INTERVALS = 2
LOCAL_ITERATIONS = 3

print(f"=== Starting Option A Training on Seed {SEED} (Width {WIDTH}, Batch {BATCH_SIZE}) ===")

# Build realization and lift
rec_s2, cfg_s2 = build_frozen_realisation(SEED)
lift_s2 = HeatLift(rec_s2, cfg_s2, modes=HEAT_MODES)

# Precompute operators
nodes = POLYNOMIAL_NODES
full_basis = np.cos(np.pi * np.linspace(0, 1, nodes)[:, None] * np.arange(HEAT_MODES))
q_grid_s2 = full_basis @ lift_s2.knots.cpu().numpy() @ full_basis.T
basis_s2 = full_basis[:, :MODES]
derivative_s2 = derivative_matrix(MODES, nodes, cfg_s2.x_max - cfg_s2.x_min)
times_s2 = torch.linspace(cfg_s2.t_min, cfg_s2.t_max, TIME_NODES, dtype=torch.float64, device="cuda")

# Amplitude bound
amplitude_s2 = (1 + Fraction(float(rec_s2["reference"][:, :, 0].abs().max()))
    + (Fraction(cfg_s2.t_max) - Fraction(cfg_s2.t_min))
    * (Fraction(float(rec_s2["forcing_u"].abs().max()))
       + abs(Fraction(cfg_s2.jump_intensity)) * Fraction(float(rec_s2["forcing_vg"].abs().max()))))

# Setup points
gen_sel = torch.Generator(device="cuda").manual_seed(360000 + SEED)
selection_points_s2 = sample_points(cfg_s2, SELECTION_DRAWS, gen_sel, "cuda")

# Audited test set
test_arc_s2 = torch.load(f"/content/benchmark/test_s{SEED}.pt", map_location="cuda", weights_only=True)
test_coll_s2 = test_arc_s2["coll"]
q_test_s2 = test_arc_s2["q"]
audited_targets_s2 = torch.load(f"/content/benchmark/n2049_f4_values_s{SEED}.pt", map_location="cuda", weights_only=True)["all"][:len(test_coll_s2)]
target_rms_s2 = audited_targets_s2.square().mean().sqrt().item()

# Model
torch.manual_seed(80000 + SEED)
base_pinn_s2 = ScalarPINN(hidden_layers=4, hidden_dim=WIDTH, num_fourier_features=64, fourier_scale=3.0, device="cuda")
system_s2 = ConditionedCorrection(base_pinn_s2, cfg_s2, hard_boundary=True).to(device="cuda", dtype=torch.float64)

# Generator
gen_train_s2 = torch.Generator(device="cuda").manual_seed(350000 + SEED)

history_s2 = []
t_start = time.time()

for iteration in range(1, PICARD_ITERATIONS + 1):
    t_iter = time.time()

    with torch.no_grad():
        coef = projected_network(system_s2, cfg_s2, times_s2, nodes=POLYNOMIAL_NODES, modes=MODES)
        target_coef = causal_picard_target(
            coef, q_grid_s2, basis_s2, derivative_s2, cfg_s2,
            damping=DAMPING, window_intervals=WINDOW_INTERVALS, local_iterations=LOCAL_ITERATIONS
        )
        target_grid = regression_grid(target_coef, modes=MODES, nodes=POLYNOMIAL_NODES, device="cuda")
        selected_target = interpolate_points(target_grid, selection_points_s2, cfg_s2)

        w_sel = system_s2(selection_points_s2[:, :1], selection_points_s2[:, 1:2], selection_points_s2[:, 2:3])
        best_loss = (w_sel - selected_target).square().mean().item()
        best_step = 0
        best_state = {k: v.detach().clone() for k, v in system_s2.state_dict().items()}

    optimizer = torch.optim.Adam(system_s2.parameters(), lr=LR)

    for step in range(1, STEPS_PER_ITER + 1):
        points = sample_points(cfg_s2, BATCH_SIZE, gen_train_s2, "cuda")
        targets = interpolate_points(target_grid, points, cfg_s2)

        optimizer.zero_grad(set_to_none=True)
        w_pred = system_s2(points[:, :1], points[:, 1:2], points[:, 2:3])
        loss = (w_pred - targets).square().mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(system_s2.parameters(), 5.0, error_if_nonfinite=True)
        optimizer.step()
        project_parameters(system_s2, WIDTH, amplitude_s2)

        if step % 100 == 0:
            with torch.no_grad():
                w_sel = system_s2(selection_points_s2[:, :1], selection_points_s2[:, 1:2], selection_points_s2[:, 2:3])
                measured = (w_sel - selected_target).square().mean().item()
                if measured < best_loss:
                    best_loss = measured
                    best_step = step
                    best_state = {k: v.detach().clone() for k, v in system_s2.state_dict().items()}

    system_s2.load_state_dict(best_state)
    elapsed = time.time() - t_iter

    # Evaluate audited error
    if iteration in (1, 5, 10, 15, 20):
        with torch.no_grad():
            w_test = system_s2(test_coll_s2[:, :1], test_coll_s2[:, 1:2], test_coll_s2[:, 2:3])
            pred = w_test + q_test_s2
            err = pred - audited_targets_s2
            rel_rms = math.sqrt(err.square().mean().item()) / target_rms_s2
        history_s2.append({"iteration": iteration, "rel_rms": rel_rms, "best_step": best_step})
        print(f"[SEED 2 | ITER {iteration:2d}/20] ({elapsed:.1f}s) Best step: {best_step:3d} | Rel RMS: {rel_rms:.4%} (Baseline w128 was: 8.564%)")
    else:
        print(f"[SEED 2 | ITER {iteration:2d}/20] ({elapsed:.1f}s) Best step: {best_step:3d} | Sel RMS: {best_loss**0.5:.6f}")

total_elapsed = time.time() - t_start
final_rel_rms_s2 = history_s2[-1]["rel_rms"]
torch.save({"model": system_s2.state_dict(), "history": history_s2, "final_rel_rms": final_rel_rms_s2}, "/content/benchmark/results_optA_s2.pt")

print("=" * 80)
print(f"SEED 2 COMPLETE in {total_elapsed:.1f}s!")
print(f"  Baseline w128 Rel RMS: 8.5644%")
print(f"  Option A w512 Rel RMS: {final_rel_rms_s2:.4%}")
print(f"  Improvement:           {0.085644 - final_rel_rms_s2:.4%}")
print("=" * 80)


=== Starting Option A Training on Seed 2 (Width 512, Batch 32768) ===
[SEED 2 | ITER  1/20] (21.9s) Best step: 600 | Rel RMS: 17.0629% (Baseline w128 was: 8.564%)
[SEED 2 | ITER  2/20] (23.6s) Best step: 600 | Sel RMS: 0.021118
[SEED 2 | ITER  3/20] (23.7s) Best step: 600 | Sel RMS: 0.019599
[SEED 2 | ITER  4/20] (23.7s) Best step: 400 | Sel RMS: 0.018588
[SEED 2 | ITER  5/20] (23.5s) Best step: 600 | Rel RMS: 8.7645% (Baseline w128 was: 8.564%)
[SEED 2 | ITER  6/20] (23.7s) Best step: 600 | Sel RMS: 0.015867
[SEED 2 | ITER  7/20] (22.6s) Best step: 600 | Sel RMS: 0.014779
[SEED 2 | ITER  8/20] (21.3s) Best step: 600 | Sel RMS: 0.014213
[SEED 2 | ITER  9/20] (21.6s) Best step: 600 | Sel RMS: 0.013069
[SEED 2 | ITER 10/20] (23.5s) Best step: 600 | Rel RMS: 7.3204% (Baseline w128 was: 8.564%)
[SEED 2 | ITER 11/20] (23.6s) Best step: 600 | Sel RMS: 0.011881
[SEED 2 | ITER 12/20] (23.7s) Best step: 400 | Sel RMS: 0.011622
[SEED 2 | ITER 13/20] (23.6s) Best step: 500 | Sel RMS: 0.011076
[SE

In [30]:
import json

summary = {
    "benchmark": "2D Stochastic Burgers + Variance-Gamma (solverfixed)",
    "protocol": "Causal Picard (causal3: 2-interval window, 3 local Picard updates, damping 0.5, 20 iterations)",
    "hardware": "NVIDIA A100-SXM4-80GB (Google Colab)",
    "arithmetic": "float64 (Double Precision Tensor Cores)",
    "comparison": {
        "seeds": [0, 1, 2],
        "baseline_w128_batch4k": {
            0: 0.060703,
            1: 0.075102,
            2: 0.085644,
            "mean": (0.060703 + 0.075102 + 0.085644) / 3
        },
        "option_a_w512_batch32k": {
            0: 0.051521,
            1: 0.046578,
            2: 0.049533,
            "mean": (0.051521 + 0.046578 + 0.049533) / 3
        },
        "absolute_reduction": {
            0: 0.060703 - 0.051521,
            1: 0.075102 - 0.046578,
            2: 0.085644 - 0.049533,
            "mean": ((0.060703 - 0.051521) + (0.075102 - 0.046578) + (0.085644 - 0.049533)) / 3
        }
    }
}

with open("/content/benchmark/final_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("=" * 85)
print("             SOLVERFIXED OPTION A CAMPAIGN RESULTS (NVIDIA A100-SXM4-80GB)            ")
print("=" * 85)
print(f"{'Driver / Seed':<15} | {'Baseline (W=128, B=4k)':<22} | {'Option A (W=512, B=32k)':<24} | {'Absolute Drop':<15}")
print("-" * 85)
for s in [0, 1, 2]:
    b = summary['comparison']['baseline_w128_batch4k'][s]
    a = summary['comparison']['option_a_w512_batch32k'][s]
    d = summary['comparison']['absolute_reduction'][s]
    print(f"Driver {s:<8} | {b:>19.4%}  | {a:>21.4%}  | {d:>12.4%}")
print("-" * 85)
mb = summary['comparison']['baseline_w128_batch4k']['mean']
ma = summary['comparison']['option_a_w512_batch32k']['mean']
md = summary['comparison']['absolute_reduction']['mean']
print(f"{'MEAN':<15} | {mb:>19.4%}  | {ma:>21.4%}  | {md:>12.4%}")
print("=" * 85)


             SOLVERFIXED OPTION A CAMPAIGN RESULTS (NVIDIA A100-SXM4-80GB)            
Driver / Seed   | Baseline (W=128, B=4k) | Option A (W=512, B=32k)  | Absolute Drop  
-------------------------------------------------------------------------------------
Driver 0        |             6.0703%  |               5.1521%  |      0.9182%
Driver 1        |             7.5102%  |               4.6578%  |      2.8524%
Driver 2        |             8.5644%  |               4.9533%  |      3.6111%
-------------------------------------------------------------------------------------
MEAN            |             7.3816%  |               4.9211%  |      2.4606%


In [1]:
import sys, torch
print("Python:", sys.version)
print("Torch:", torch.__version__, "CUDA:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0))
# Check if key functions exist in globals
funcs = ['ScalarPINN', 'ConditionedCorrection', 'HeatLift', 'causal_picard_target', 'build_frozen_realisation', 'cfg']
for fn in funcs:
    print(f"{fn} in globals():", fn in globals())


Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Torch: 2.11.0+cu130 CUDA: True
Device: NVIDIA A100-SXM4-80GB
ScalarPINN in globals(): False
ConditionedCorrection in globals(): False
HeatLift in globals(): False
causal_picard_target in globals(): False
build_frozen_realisation in globals(): False
cfg in globals(): False


In [2]:
import os
print("CWD files:", os.listdir("/content"))
if os.path.exists("/content/benchmark"):
    print("Benchmark files:", sorted(os.listdir("/content/benchmark")))


CWD files: ['.config', 'sample_data']


In [3]:
import os, sys, math, time, dataclasses, hashlib, json
from dataclasses import dataclass
from fractions import Fraction
import numpy as np
import scipy
from scipy.linalg import solve_banded
from scipy.fft import dctn
import torch
import torch.nn as nn
import torch.nn.functional as F

print(f"PyTorch version: {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

def tensor_sha(tensor):
    return hashlib.sha256(tensor.detach().cpu().contiguous().numpy().tobytes()).hexdigest()

@dataclass
class ConfigBurgersVG2D:
    x_min: float = -1.0
    x_max: float =  1.0
    y_min: float = -1.0
    y_max: float =  1.0
    t_min: float =  0.0
    t_max: float =  0.5
    Nx: int = 64
    Ny: int = 64
    Nt: int = 512
    nu_u: float = 1.0e-2
    sigma_u: float = 0.05
    noise_type_u: str = "white"
    hurst_u: float = 0.7
    sigma_vg: float = 0.10
    nu_vg: float = 0.20
    theta_vg: float = -0.03
    jump_intensity: float = 0.5
    ic_amplitude: float = 0.6
    burn_t: float = 1.0
    hidden_layers: int = 4
    hidden_dim: int = 128
    num_fourier_features: int = 64
    fourier_scale: float = 3.0

    @property
    def dx(self): return (self.x_max - self.x_min) / (self.Nx - 1)
    @property
    def dy(self): return (self.y_max - self.y_min) / (self.Ny - 1)
    @property
    def dt(self): return (self.t_max - self.t_min) / (self.Nt - 1)

def generate_vg_jump_field(Nx, Ny, Nt, dt, sigma_vg=0.10, nu_vg=0.20, theta_vg=-0.03, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    out = np.zeros((Nx, Ny, Nt), dtype=np.float32)
    for k in range(Nt):
        G = rng.gamma(shape=dt / nu_vg, scale=nu_vg, size=(Nx, Ny))
        Z = rng.normal(0.0, 1.0, size=(Nx, Ny))
        out[:, :, k] = (theta_vg * G + sigma_vg * np.sqrt(np.maximum(G, 1e-12)) * Z)
    return out

def _tridiag_neumann(N, r):
    half_r = 0.5 * r
    diag = np.full(N, 1 + 2 * half_r); off = np.full(N - 1, -half_r)
    diag[0] = 1 + half_r; diag[-1] = 1 + half_r
    lhs = np.zeros((3, N))
    lhs[0, 1:] = off; lhs[1, :] = diag; lhs[2, :-1] = off
    d_rhs = np.full(N, 1 - 2 * half_r)
    d_rhs[0] = 1 - half_r; d_rhs[-1] = 1 - half_r
    o_rhs = half_r
    def rhs_mult(u):
        v = d_rhs * u
        v[:-1] += o_rhs * u[1:]; v[1:] += o_rhs * u[:-1]
        return v
    return lhs, rhs_mult

def _adi_step(u, D, dx, dy, dt, forcing):
    Nx, Ny = u.shape
    rx = D * dt / dx ** 2; ry = D * dt / dy ** 2
    lhs_x, rhs_x = _tridiag_neumann(Nx, rx)
    lhs_y, rhs_y = _tridiag_neumann(Ny, ry)
    u_half = np.empty_like(u)
    for j in range(Ny):
        col = u[:, j]
        if 0 < j < Ny - 1:
            yyc = ry * 0.5 * (u[:, j+1] - 2*u[:, j] + u[:, j-1])
        elif j == 0:
            yyc = ry * 0.5 * (u[:, j+1] - u[:, j])
        else:
            yyc = ry * 0.5 * (u[:, j-1] - u[:, j])
        rhs_vec = col + yyc + 0.5 * dt * forcing[:, j]
        u_half[:, j] = solve_banded((1, 1), lhs_x, rhs_vec)
    u_new = np.empty_like(u)
    for i in range(Nx):
        row = u_half[i, :]
        if 0 < i < Nx - 1:
            xxc = rx * 0.5 * (u_half[i+1, :] - 2*u_half[i, :] + u_half[i-1, :])
        elif i == 0:
            xxc = rx * 0.5 * (u_half[i+1, :] - u_half[i, :])
        else:
            xxc = rx * 0.5 * (u_half[i-1, :] - u_half[i, :])
        rhs_vec = row + xxc + 0.5 * dt * forcing[i, :]
        u_new[i, :] = solve_banded((1, 1), lhs_y, rhs_vec)
    return u_new

def _upwind_advection(u, dx, dy):
    pos = (u >= 0)
    bwd_x = np.zeros_like(u); fwd_x = np.zeros_like(u)
    bwd_x[1:, :] = (u[1:, :] - u[:-1, :]) / dx
    fwd_x[:-1, :] = (u[1:, :] - u[:-1, :]) / dx
    u_x = np.where(pos, bwd_x, fwd_x)
    bwd_y = np.zeros_like(u); fwd_y = np.zeros_like(u)
    bwd_y[:, 1:] = (u[:, 1:] - u[:, :-1]) / dy
    fwd_y[:, :-1] = (u[:, 1:] - u[:, :-1]) / dy
    u_y = np.where(pos, bwd_y, fwd_y)
    return u * (u_x + u_y)

def _ic_u(X, Y, cfg, rng):
    base = cfg.ic_amplitude * np.sin(np.pi * X) * np.sin(np.pi * Y)
    perturb = 0.05 * rng.standard_normal(X.shape)
    return base + perturb

def build_frozen_realisation(seed, device="cuda"):
    cfg = ConfigBurgersVG2D(t_min=0.0, t_max=0.5, Nt=512)
    burn_t = 1.0
    dt0 = cfg.dt
    n_burn = int(round(burn_t / dt0))
    Nt_gen = cfg.Nt + n_burn
    t_max_gen = (Nt_gen - 1) * dt0
    gen = dataclasses.replace(cfg, t_min=0.0, Nt=Nt_gen, t_max=t_max_gen)

    rng = np.random.default_rng(seed)
    raw = rng.standard_normal((gen.Nx, gen.Ny, gen.Nt))
    noise = ((raw.astype(np.float32) * np.float32(gen.sigma_u)) / np.float32(np.sqrt(gen.dx * gen.dy * gen.dt)))

    rng_vg = np.random.default_rng(seed + 10000)
    vg_inc = generate_vg_jump_field(
        gen.Nx, gen.Ny, gen.Nt, gen.dt,
        sigma_vg=gen.sigma_vg, nu_vg=gen.nu_vg, theta_vg=gen.theta_vg, rng=rng_vg)

    vg_rate = vg_inc / gen.dt
    x_grid = np.linspace(gen.x_min, gen.x_max, gen.Nx)
    y_grid = np.linspace(gen.y_min, gen.y_max, gen.Ny)
    t_grid = np.linspace(gen.t_min, gen.t_max, gen.Nt)
    X, Yg = np.meshgrid(x_grid, y_grid, indexing="ij")
    U = np.zeros((gen.Nx, gen.Ny, gen.Nt), dtype=np.float32)
    U[:, :, 0] = _ic_u(X, Yg, gen, rng)

    for n in range(gen.Nt - 1):
        u_n = U[:, :, n].astype(np.float64)
        advect = _upwind_advection(u_n, gen.dx, gen.dy)
        forcing = -advect + noise[:, :, n] + gen.jump_intensity * vg_rate[:, :, n]
        u_np1 = _adi_step(u_n, gen.nu_u, gen.dx, gen.dy, gen.dt, forcing)
        u_np1 = np.clip(u_np1, -3.0, 3.0)
        U[:, :, n + 1] = u_np1.astype(np.float32)

    noise = noise[:, :, n_burn:]
    vg_rate = vg_rate[:, :, n_burn:]
    U = U[:, :, n_burn:]
    t_grid = t_grid[n_burn:]

    record = {
        "reference": torch.tensor(U, dtype=torch.float64, device=device),
        "forcing_u": torch.tensor(noise, dtype=torch.float32, device=device).double(),
        "forcing_vg": torch.tensor(vg_rate, dtype=torch.float32, device=device).double(),
        "x_grid": torch.tensor(x_grid, dtype=torch.float64, device=device),
        "y_grid": torch.tensor(y_grid, dtype=torch.float64, device=device),
        "t_grid": torch.tensor(t_grid, dtype=torch.float64, device=device),
    }
    target_cfg = dataclasses.replace(cfg, t_min=float(t_grid[0]), t_max=float(t_grid[-1]))
    return record, target_cfg

class FourierFeatures(nn.Module):
    def __init__(self, num_features: int = 64, scale: float = 3.0, in_dim: int = 3, device="cuda"):
        super().__init__()
        B = scale * torch.randn(in_dim, num_features, dtype=torch.float64, device=device)
        self.register_buffer("B", B)

    def forward(self, x, y, t):
        v = torch.cat([x, y, t], dim=-1)
        proj = 2.0 * math.pi * v @ self.B
        return torch.cat([torch.sin(proj), torch.cos(proj)], dim=-1)

class ScalarPINN(nn.Module):
    def __init__(self, hidden_layers: int = 4, hidden_dim: int = 512,
                 num_fourier_features: int = 64, fourier_scale: float = 3.0, device="cuda"):
        super().__init__()
        self.fourier = FourierFeatures(num_fourier_features, fourier_scale, in_dim=3, device=device)
        in_dim = 2 * num_fourier_features
        layers = [nn.Linear(in_dim, hidden_dim, dtype=torch.float64, device=device), nn.Tanh()]
        for _ in range(hidden_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim, dtype=torch.float64, device=device), nn.Tanh()]
        head = nn.Linear(hidden_dim, 1, dtype=torch.float64, device=device)
        with torch.no_grad():
            head.weight.zero_()
            head.bias.zero_()
        layers.append(head)
        self.net = nn.Sequential(*layers)

    def forward(self, x, y, t):
        feat = self.fourier(x, y, t)
        return self.net(feat)

class ConditionedCorrection(nn.Module):
    def __init__(self, base: ScalarPINN, cfg, hard_boundary: bool = True):
        super().__init__()
        self.base = base
        self.cfg = cfg
        self.hard_boundary = hard_boundary

    def forward(self, x, y, t):
        if self.hard_boundary:
            x_b = torch.cos(math.pi * (x - self.cfg.x_min) / (self.cfg.x_max - self.cfg.x_min))
            y_b = torch.cos(math.pi * (y - self.cfg.y_min) / (self.cfg.y_max - self.cfg.y_min))
        else:
            x_b, y_b = x, y
        value = self.base(x_b, y_b, t)
        envelope = (t - self.cfg.t_min) / (self.cfg.t_max - self.cfg.t_min)
        return envelope * value

def project_parameters(model, width, amplitude):
    with torch.no_grad():
        for p in model.parameters():
            p.clamp_(-width, width)
        head = model.base.net[-1]
        total = head.weight.abs().sum() + head.bias.abs().sum()
        limit = float(amplitude - Fraction(5, 8))
        if total.item() > limit:
            factor = limit / total
            head.weight.mul_(factor)
            head.bias.mul_(factor)

def hat_coefficients(nodes, modes, length, device="cpu"):
    h = length / (nodes - 1)
    k = torch.arange(modes, dtype=torch.float64, device=device)
    omega = k * math.pi / length
    omega_safe = omega.clone()
    omega_safe[0] = 1
    scale = 2 * torch.sin(0.5 * omega * h).square() / (h * omega_safe.square())
    x = torch.arange(nodes, dtype=torch.float64, device=device) * h
    coefficients = 2 * scale[:, None] * torch.cos(omega[:, None] * x[None])
    coefficients[:, [0, -1]] *= 0.5
    coefficients[0] = h
    coefficients[0, [0, -1]] = 0.5 * h
    coefficients[1:] *= 2 / length
    coefficients[0] /= length
    return coefficients

def modal_weights(eigenvalues, dt):
    z = eigenvalues * dt
    safe = eigenvalues.clone()
    safe[safe == 0] = 1
    a = -torch.expm1(-z) / safe
    series = dt**2 * (0.5 - z/6 + z.square()/24 - z**3/120 + z**4/720 - z**5/5040)
    b = torch.where(z.abs() < 0.01, series, (dt - a) / safe)
    a = torch.where(eigenvalues == 0, torch.full_like(a, dt), a)
    return torch.exp(-z), a, b

class HeatLift:
    def __init__(self, record, cfg, modes=128):
        self.cfg = cfg
        self.modes = modes
        rate = record["forcing_u"] + cfg.jump_intensity * record["forcing_vg"]
        ax = hat_coefficients(rate.shape[0], modes, cfg.x_max - cfg.x_min, rate.device)
        ay = hat_coefficients(rate.shape[1], modes, cfg.y_max - cfg.y_min, rate.device)
        self.forcing = torch.einsum("ki,ijt,lj->tkl", ax, rate, ay).contiguous()
        self.initial = ax @ record["reference"][:, :, 0] @ ay.T
        k = torch.arange(modes, dtype=torch.float64, device=rate.device)
        self.wx = k * math.pi / (cfg.x_max - cfg.x_min)
        self.wy = k * math.pi / (cfg.y_max - cfg.y_min)
        self.lam = cfg.nu_u * (self.wx[:, None].square() + self.wy[None].square())
        self.dt = (cfg.t_max - cfg.t_min) / (len(self.forcing) - 1)
        decay, a, b = modal_weights(self.lam, self.dt)
        self.knots = torch.empty_like(self.forcing)
        self.knots[0] = self.initial
        for j in range(len(self.forcing) - 1):
            self.knots[j + 1] = decay * self.knots[j] + a * self.forcing[j] + (b / self.dt) * (self.forcing[j + 1] - self.forcing[j])

    def evaluate(self, points):
        index = ((points[:, 2] - self.cfg.t_min) / self.dt).floor().long().clamp(0, len(self.forcing) - 2)
        elapsed = (points[:, 2] - self.cfg.t_min - index.to(points.dtype) * self.dt).view(-1, 1, 1)
        safe = self.lam.clone()
        safe[safe == 0] = 1
        z = self.lam * elapsed
        a = -torch.expm1(-z) / safe
        a = torch.where(self.lam == 0, elapsed, a)
        series = elapsed.square() * (0.5 - z/6 + z.square()/24 - z**3/120 + z**4/720 - z**5/5040)
        b = torch.where(z.abs() < 0.01, series, (elapsed - a) / safe)
        left, right = self.forcing[index], self.forcing[index + 1]
        coefficients = torch.exp(-z) * self.knots[index] + a * left + (b / self.dt) * (right - left)
        px = (points[:, :1] - self.cfg.x_min) * self.wx
        py = (points[:, 1:2] - self.cfg.y_min) * self.wy
        cx, cy = px.cos(), py.cos()
        q = torch.einsum("bi,bij,bj->b", cx, coefficients, cy).view(-1, 1)
        return q

def derivative_matrix(modes, square_modes, length):
    k = np.arange(modes)[:, None]
    m = np.arange(square_modes)[None, :]
    plus, minus = m + k, m - k
    def fraction(n):
        return np.divide(2 * (n % 2 != 0), n, out=np.zeros_like(n, dtype=float), where=n != 0)
    normal = np.where(k == 0, 1.0, 2.0)
    return -normal * m / (4 * length) * (fraction(plus) + fraction(minus))

def cosine_coefficients(values):
    assert values.shape[-2] == values.shape[-1]
    result = dctn(values, type=1, axes=(-2, -1), workers=4) / (values.shape[-1] - 1) ** 2
    result[..., (0, -1), :] *= 0.5
    result[..., :, (0, -1)] *= 0.5
    return result

def nonlinear_source(q, w, basis, derivative, chunk=32):
    modes = w.shape[-1]
    result = np.empty_like(w)
    for start in range(0, len(w), chunk):
        stop = min(start + chunk, len(w))
        values = q[start:stop] + basis @ w[start:stop] @ basis.T
        with np.errstate(over="ignore", invalid="ignore"):
            square = cosine_coefficients(values * values)
            result[start:stop] = (derivative @ square[:, :, :modes]
                                  + square[:, :modes, :] @ derivative.T)
    return result

def response_ode(source, initial, nu, lx, ly, dt):
    modes = source.shape[-1]
    k = np.arange(modes)
    lam = nu * ((np.pi * k[:, None] / lx) ** 2 + (np.pi * k[None, :] / ly) ** 2)
    z = lam * dt
    decay = np.exp(-z)
    safe = np.where(z == 0, 1, z)
    a = -np.expm1(-z) / safe
    right = (z + np.expm1(-z)) / safe ** 2
    small = np.abs(z) < 0.01
    right = np.where(small, 0.5 - z / 6 + z ** 2 / 24 - z ** 3 / 120 + z ** 4 / 720 - z ** 5 / 5040, right)
    a = np.where(z == 0, 1, a)
    left, right = dt * (a - right), dt * right
    output = np.empty_like(source)
    output[0] = initial
    with np.errstate(over="ignore", invalid="ignore"):
        for j in range(len(source) - 1):
            output[j + 1] = decay * output[j] + left * source[j] + right * source[j + 1]
    return output

def causal_picard_target(neural, q, basis, derivative, cfg, damping=0.5, window_intervals=2, local_iterations=3):
    count, modes = neural.shape[:2]
    dt = (cfg.t_max - cfg.t_min) / (count - 1)
    initial = np.zeros((modes, modes))
    lam = cfg.nu_u * ((np.pi * np.arange(modes)[:, None] / (cfg.x_max - cfg.x_min)) ** 2
                      + (np.pi * np.arange(modes)[None, :] / (cfg.y_max - cfg.y_min)) ** 2)
    result = np.empty_like(neural)
    for start in range(0, count - 1, window_intervals):
        stop = min(start + window_intervals, count - 1)
        decay = np.exp(-np.arange(stop - start + 1)[:, None, None] * dt * lam)
        waveform = neural[start:stop + 1] + decay * (initial - neural[start])
        for _ in range(local_iterations):
            source = nonlinear_source(q[start:stop + 1], waveform, basis, derivative)
            h = response_ode(-source, initial, cfg.nu_u, cfg.x_max - cfg.x_min, cfg.y_max - cfg.y_min, dt)
            waveform = (1 - damping) * waveform + damping * h
        waveform[0] = initial
        result[start:stop + 1] = waveform
        initial = waveform[-1].copy()
    return result

def network_grid(system, cfg, times, nodes=257, chunk=16):
    x = torch.linspace(cfg.x_min, cfg.x_max, nodes, dtype=times.dtype, device=times.device)
    y = torch.linspace(cfg.y_min, cfg.y_max, nodes, dtype=times.dtype, device=times.device)
    X, Y = torch.meshgrid(x, y, indexing="ij")
    output = times.new_empty(len(times), nodes, nodes)
    with torch.no_grad():
        for start in range(0, len(times), chunk):
            count = min(chunk, len(times) - start)
            xx = X[None].expand(count, -1, -1).reshape(-1, 1)
            yy = Y[None].expand(count, -1, -1).reshape(-1, 1)
            tt = times[start:start+count, None, None].expand(-1, nodes, nodes).reshape(-1, 1)
            output[start:start+count] = system(xx, yy, tt).view(count, nodes, nodes)
    return output

def projected_network(system, cfg, times, nodes=257, modes=128):
    values = network_grid(system, cfg, times, nodes=nodes).cpu().numpy()
    coefficients = cosine_coefficients(values)[:, :modes, :modes].copy()
    return coefficients

def regression_grid(coefficients, modes=128, nodes=257, device="cuda"):
    coef = torch.from_numpy(coefficients).to(device)
    z = torch.linspace(0, 1, nodes, dtype=torch.float64, device=device)
    k = torch.arange(modes, dtype=torch.float64, device=device)
    basis = (torch.pi * z[:, None] * k[None]).cos()
    output = coef.new_empty(len(coef), nodes, nodes)
    for start in range(0, len(coef), 16):
        count = min(16, len(coef) - start)
        output[start:start+count] = torch.einsum("xi,bij,yj->bxy", basis, coef[start:start+count], basis)
    return output.permute(1, 2, 0).contiguous()

def interpolate_points(field, points, cfg):
    lengths = field.new_tensor([cfg.x_max - cfg.x_min, cfg.y_max - cfg.y_min, cfg.t_max - cfg.t_min])
    lower = field.new_tensor([cfg.x_min, cfg.y_min, cfg.t_min])
    sizes = field.new_tensor(field.shape) - 1
    q = ((points - lower) / lengths * sizes).clamp(min=0)
    q = torch.minimum(q, sizes)
    left = q.floor().long()
    right = torch.minimum(left + 1, sizes.long())
    weight = q - left
    result = points.new_zeros(len(points))
    for a in (0, 1):
        for b in (0, 1):
            for c in (0, 1):
                indices = [right[:, i] if side else left[:, i] for i, side in enumerate((a, b, c))]
                factor = torch.ones_like(result)
                for i, side in enumerate((a, b, c)):
                    factor = factor * (weight[:, i] if side else 1 - weight[:, i])
                result += factor * field[tuple(indices)]
    return result.view(-1, 1)

def sample_points(cfg, count, generator, device="cuda"):
    lo = torch.tensor([cfg.x_min, cfg.y_min, cfg.t_min], dtype=torch.float64, device=device)
    length = lo.new_tensor([cfg.x_max - cfg.x_min, cfg.y_max - cfg.y_min, cfg.t_max - cfg.t_min])
    return lo + length * torch.rand(count, 3, generator=generator, device=device, dtype=torch.float64)

# Download audited benchmark files
os.makedirs("/content/benchmark", exist_ok=True)
if not os.path.exists("/content/benchmark/test.pt"):
    print("Downloading test.pt...")
    os.system("curl -s -o /content/benchmark/test.pt https://litter.catbox.moe/lxq3vs.pt")
if not os.path.exists("/content/benchmark/n2049_f4_values.pt"):
    print("Downloading n2049_f4_values.pt...")
    os.system("curl -s -o /content/benchmark/n2049_f4_values.pt https://litter.catbox.moe/32sy5y.pt")
if not os.path.exists("/content/benchmark/baseline_model.pt"):
    print("Downloading baseline_model.pt...")
    os.system("curl -s -o /content/benchmark/baseline_model.pt https://litter.catbox.moe/i6pmzw.pt")

# Verify Driver 0 hashes
t0 = time.time()
rec0, cfg0 = build_frozen_realisation(0)
EXPECTED_0 = {
    "u": "ab152391e1e9181b3fd307edcaa83b29f942ecfbb8b2795a00064b8ed5c8b99b",
    "vg": "c61af0a64eb07c5f519b061e65af13269c843e325d4e536f0464b96804061045",
    "ref": "3ef7f4d716c49f5ef7c4569be7a59d0bec24b603d79f1e1f22183ce2f23c9e18",
}
assert tensor_sha(rec0["forcing_u"]) == EXPECTED_0["u"], "Driver 0 forcing_u mismatch!"
assert tensor_sha(rec0["forcing_vg"]) == EXPECTED_0["vg"], "Driver 0 forcing_vg mismatch!"
assert tensor_sha(rec0["reference"]) == EXPECTED_0["ref"], "Driver 0 reference mismatch!"
print(f"Driver 0 built and verified bit-for-bit exact in {time.time()-t0:.2f}s!")


PyTorch version: 2.11.0+cu130, CUDA available: True
Device: NVIDIA A100-SXM4-80GB, VRAM: 85.09 GB
Driver 0 built and verified bit-for-bit exact in 9.62s!


In [12]:
# ==============================================================================
# OPTION 2: Ultra-Capacity & Modal Scaling (W=1024, K=256 modes, 3.3M parameters)
# Target: Driver 0 (Seed 0) against Continuous ETD2 Reference (n2049_f4_values.pt)
# ==============================================================================

import gc
gc.collect()
torch.cuda.empty_cache()

SEED = 0
WIDTH = 1024
BATCH_SIZE = 32768
SELECTION_DRAWS = 32768
PICARD_ITERATIONS = 20
STEPS_PER_ITER = 600
LR = 0.001
MODES = 256
HEAT_MODES = 256
POLYNOMIAL_NODES = 513
TIME_NODES = 512
DAMPING = 0.5
WINDOW_INTERVALS = 2
LOCAL_ITERATIONS = 3

print("=" * 80)
print(f"STARTING OPTION 2: WIDTH {WIDTH}, MODES {MODES}, NODES {POLYNOMIAL_NODES}")
print(f"Hardware: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM)")
print("=" * 80)

# 1. Build realization and lift with 256 modes
t0 = time.time()
rec0, cfg0 = build_frozen_realisation(SEED)
lift0_256 = HeatLift(rec0, cfg0, modes=HEAT_MODES)
print(f"Heat lift with {HEAT_MODES} modes computed in {time.time()-t0:.2f}s")

# 2. Precompute modal operators
nodes = POLYNOMIAL_NODES
full_basis = np.cos(np.pi * np.linspace(0, 1, nodes)[:, None] * np.arange(HEAT_MODES))
t0 = time.time()
q_grid0 = full_basis @ lift0_256.knots.cpu().numpy() @ full_basis.T
print(f"q_grid ({q_grid0.shape}, {q_grid0.nbytes/1e6:.1f} MB) built in {time.time()-t0:.2f}s")

basis0 = full_basis[:, :MODES]
derivative0 = derivative_matrix(MODES, nodes, cfg0.x_max - cfg0.x_min)
times0 = torch.linspace(cfg0.t_min, cfg0.t_max, TIME_NODES, dtype=torch.float64, device="cuda")

# 3. Amplitude bound
amplitude0 = (1 + Fraction(float(rec0["reference"][:, :, 0].abs().max()))
    + (Fraction(cfg0.t_max) - Fraction(cfg0.t_min))
    * (Fraction(float(rec0["forcing_u"].abs().max()))
       + abs(Fraction(cfg0.jump_intensity)) * Fraction(float(rec0["forcing_vg"].abs().max()))))

# 4. Selection & Test Points
gen_sel = torch.Generator(device="cuda").manual_seed(360000 + SEED)
selection_points0 = sample_points(cfg0, SELECTION_DRAWS, gen_sel, "cuda")

test_arc0 = torch.load("/content/benchmark/test.pt", map_location="cuda", weights_only=False)
test_coll0 = test_arc0["coll"]
audited_targets0 = torch.load("/content/benchmark/n2049_f4_values.pt", map_location="cuda", weights_only=False)["all"][:len(test_coll0)]
target_rms0 = audited_targets0.square().mean().sqrt().item()

# Compute exact q on test points with 256 modes
q_test0 = lift0_256.evaluate(test_coll0)
q_baseline_err = (q_test0 - audited_targets0).square().mean().sqrt().item() / target_rms0
print(f"Lift-only relative RMS error on audited test set: {q_baseline_err:.4%}")

# 5. Initialize Model (Width 1024, 4 layers, ~3.3M params)
torch.manual_seed(80000 + SEED)
base_pinn0 = ScalarPINN(hidden_layers=4, hidden_dim=WIDTH, num_fourier_features=64, fourier_scale=3.0, device="cuda")
num_params = sum(p.numel() for p in base_pinn0.parameters())
print(f"ScalarPINN initialized: {num_params:,} parameters (Float64)")

system0 = ConditionedCorrection(base_pinn0, cfg0, hard_boundary=True).to(device="cuda", dtype=torch.float64)
gen_train0 = torch.Generator(device="cuda").manual_seed(350000 + SEED)

# 6. Training Loop (20 Picard Iterations)
history_opt2 = []
t_start = time.time()

for iteration in range(1, PICARD_ITERATIONS + 1):
    t_iter = time.time()

    with torch.no_grad():
        coef = projected_network(system0, cfg0, times0, nodes=POLYNOMIAL_NODES, modes=MODES)
        target_coef = causal_picard_target(
            coef, q_grid0, basis0, derivative0, cfg0,
            damping=DAMPING, window_intervals=WINDOW_INTERVALS, local_iterations=LOCAL_ITERATIONS
        )
        target_grid = regression_grid(target_coef, modes=MODES, nodes=POLYNOMIAL_NODES, device="cuda")
        selected_target = interpolate_points(target_grid, selection_points0, cfg0)

        w_sel = system0(selection_points0[:, :1], selection_points0[:, 1:2], selection_points0[:, 2:3])
        best_loss = (w_sel - selected_target).square().mean().item()
        best_step = 0
        best_state = {k: v.detach().clone() for k, v in system0.state_dict().items()}

    optimizer = torch.optim.Adam(system0.parameters(), lr=LR)

    for step in range(1, STEPS_PER_ITER + 1):
        points = sample_points(cfg0, BATCH_SIZE, gen_train0, "cuda")
        targets = interpolate_points(target_grid, points, cfg0)

        optimizer.zero_grad(set_to_none=True)
        w_pred = system0(points[:, :1], points[:, 1:2], points[:, 2:3])
        loss = (w_pred - targets).square().mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(system0.parameters(), 5.0, error_if_nonfinite=True)
        optimizer.step()
        project_parameters(system0, WIDTH, amplitude0)

        if step % 100 == 0:
            with torch.no_grad():
                w_sel = system0(selection_points0[:, :1], selection_points0[:, 1:2], selection_points0[:, 2:3])
                measured = (w_sel - selected_target).square().mean().item()
                if measured < best_loss:
                    best_loss = measured
                    best_step = step
                    best_state = {k: v.detach().clone() for k, v in system0.state_dict().items()}

    system0.load_state_dict(best_state)
    elapsed = time.time() - t_iter

    # Audited error evaluation at milestones
    if iteration in (1, 5, 10, 15, 20):
        with torch.no_grad():
            w_test = system0(test_coll0[:, :1], test_coll0[:, 1:2], test_coll0[:, 2:3])
            pred = w_test + q_test0
            err = pred - audited_targets0
            rel_rms = math.sqrt(err.square().mean().item()) / target_rms0
        history_opt2.append({"iteration": iteration, "rel_rms": rel_rms, "best_step": best_step, "elapsed": elapsed})
        print(f"[OPT 2 (W=1024, K=256) | ITER {iteration:2d}/20] ({elapsed:.1f}s) Best step: {best_step:3d} | Rel RMS: {rel_rms:.4%} (Baseline: 6.0703%, Opt A: 5.1521%)")
    else:
        print(f"[OPT 2 (W=1024, K=256) | ITER {iteration:2d}/20] ({elapsed:.1f}s) Best step: {best_step:3d} | Sel RMS: {best_loss**0.5:.6f}")

total_elapsed = time.time() - t_start
final_rel_rms_opt2 = history_opt2[-1]["rel_rms"]
torch.save({"model": system0.state_dict(), "history": history_opt2, "final_rel_rms": final_rel_rms_opt2}, "/content/benchmark/results_opt2_w1024_k256_s0.pt")

print("=" * 80)
print(f"OPTION 2 COMPLETE in {total_elapsed:.1f}s ({total_elapsed/60:.2f} min)!")
print(f"  Baseline (W=128, K=128) Rel RMS: 6.0703%")
print(f"  Option A (W=512, K=128) Rel RMS: 5.1521%")
print(f"  Option 2 (W=1024,K=256) Rel RMS: {final_rel_rms_opt2:.4%}")
print(f"  Total Reduction vs Baseline:     {0.060703 - final_rel_rms_opt2:.4%}")
print(f"  Total Reduction vs Option A:     {0.051521 - final_rel_rms_opt2:.4%}")
print("=" * 80)


STARTING OPTION 2: WIDTH 1024, MODES 256, NODES 513
Hardware: NVIDIA A100-SXM4-80GB (85.1 GB VRAM)
Heat lift with 256 modes computed in 9.24s
q_grid ((512, 513, 513), 1077.9 MB) built in 3.22s
Lift-only relative RMS error on audited test set: 20.5148%
ScalarPINN initialized: 3,281,921 parameters (Float64)
[OPT 2 (W=1024, K=256) | ITER  1/20] (123.8s) Best step: 600 | Rel RMS: 9.2558% (Baseline: 6.0703%, Opt A: 5.1521%)
[OPT 2 (W=1024, K=256) | ITER  2/20] (121.7s) Best step: 600 | Sel RMS: 0.012043
[OPT 2 (W=1024, K=256) | ITER  3/20] (121.6s) Best step: 500 | Sel RMS: 0.011279
[OPT 2 (W=1024, K=256) | ITER  4/20] (121.7s) Best step: 500 | Sel RMS: 0.010904
[OPT 2 (W=1024, K=256) | ITER  5/20] (121.2s) Best step: 600 | Rel RMS: 6.6345% (Baseline: 6.0703%, Opt A: 5.1521%)
[OPT 2 (W=1024, K=256) | ITER  6/20] (120.4s) Best step:   0 | Sel RMS: 0.010174
[OPT 2 (W=1024, K=256) | ITER  7/20] (120.2s) Best step: 600 | Sel RMS: 0.010116
[OPT 2 (W=1024, K=256) | ITER  8/20] (120.4s) Best step:

In [6]:
!head -n 5 /content/benchmark/test.pt
!ls -lh /content/benchmark/


<!doctype html><html lang="en"><meta charset="UTF-8"><meta content="nofollow,noarchive,noindex" name="robots"><title>404 | Not Found</title><meta content="width=device-width,initial-scale=1" name="viewport"><meta content="IE=edge" http-equiv="X-UA-Compatible"><meta content="404 | Not Found" name="title"><meta content="The server cannot find the requested page." name="description"><meta content="404 | Not Found" property="og:title"><meta content="The server cannot find the requested page." property="og:description"><meta content="404 | Not Found" property="twitter:title"><meta content="The server cannot find the requested page." property="twitter:description"><link href="data:image/svg+xml, %3Csvg version='1.0' xmlns='http://www.w3.org/2000/svg' width='96.000000pt' height='96.000000pt' viewBox='0 0 96.000000 96.000000' preserveAspectRatio='xMidYMid meet'%3E%3Cg transform='translate(0.000000,96.000000) scale(0.100000,-0.100000)'%0Afill='%23085577' stroke='none'%3E%3Cpath d='M535 863 c-22

In [7]:
!rm -f /content/benchmark/test.pt /content/benchmark/n2049_f4_values.pt
!curl -s -o /content/benchmark/test.pt https://files.catbox.moe/rg0388.pt
!curl -s -o /content/benchmark/n2049_f4_values.pt https://files.catbox.moe/6e1u44.pt
!sha256sum /content/benchmark/test.pt /content/benchmark/n2049_f4_values.pt


4d2e720bbf0a1bcf7e78a3b812fb67ea524500c513fbe1e468e81bffb198edf3  /content/benchmark/test.pt
5cc9f975906160126fc7c5d055fe98d194f042cf99c841d4a763d8689fe870e1  /content/benchmark/n2049_f4_values.pt


In [9]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()
print("GPU memory cleared. Free VRAM:", f"{torch.cuda.mem_get_info()[0]/1e9:.2f} GB / {torch.cuda.mem_get_info()[1]/1e9:.2f} GB")

# Update HeatLift.evaluate with chunking to support any number of points and high mode counts (K=256)
def chunked_evaluate(self, points, chunk=4096):
    out = []
    for start in range(0, len(points), chunk):
        pts = points[start:start+chunk]
        index = ((pts[:, 2] - self.cfg.t_min) / self.dt).floor().long().clamp(0, len(self.forcing) - 2)
        elapsed = (pts[:, 2] - self.cfg.t_min - index.to(pts.dtype) * self.dt).view(-1, 1, 1)
        safe = self.lam.clone()
        safe[safe == 0] = 1
        z = self.lam * elapsed
        a = -torch.expm1(-z) / safe
        a = torch.where(self.lam == 0, elapsed, a)
        series = elapsed.square() * (0.5 - z/6 + z.square()/24 - z**3/120 + z**4/720 - z**5/5040)
        b = torch.where(z.abs() < 0.01, series, (elapsed - a) / safe)
        left, right = self.forcing[index], self.forcing[index + 1]
        coefficients = torch.exp(-z) * self.knots[index] + a * left + (b / self.dt) * (right - left)
        px = (pts[:, :1] - self.cfg.x_min) * self.wx
        py = (pts[:, 1:2] - self.cfg.y_min) * self.wy
        cx, cy = px.cos(), py.cos()
        q = torch.einsum("bi,bij,bj->b", cx, coefficients, cy).view(-1, 1)
        out.append(q)
    return torch.cat(out, dim=0)

HeatLift.evaluate = chunked_evaluate
print("HeatLift.evaluate updated with chunked evaluation.")


GPU memory cleared. Free VRAM: 49.58 GB / 85.09 GB
HeatLift.evaluate updated with chunked evaluation.


In [11]:
import gc, torch
gc.collect()
torch.cuda.empty_cache()

def network_grid(system, cfg, times, nodes=257, chunk=2):
    x = torch.linspace(cfg.x_min, cfg.x_max, nodes, dtype=times.dtype, device=times.device)
    y = torch.linspace(cfg.y_min, cfg.y_max, nodes, dtype=times.dtype, device=times.device)
    X, Y = torch.meshgrid(x, y, indexing="ij")
    output = times.new_empty(len(times), nodes, nodes)
    with torch.no_grad():
        for start in range(0, len(times), chunk):
            count = min(chunk, len(times) - start)
            xx = X[None].expand(count, -1, -1).reshape(-1, 1)
            yy = Y[None].expand(count, -1, -1).reshape(-1, 1)
            tt = times[start:start+count, None, None].expand(-1, nodes, nodes).reshape(-1, 1)
            output[start:start+count] = system(xx, yy, tt).view(count, nodes, nodes)
    return output

def projected_network(system, cfg, times, nodes=257, modes=128, chunk=2):
    values = network_grid(system, cfg, times, nodes=nodes, chunk=chunk).cpu().numpy()
    coefficients = cosine_coefficients(values)[:, :modes, :modes].copy()
    return coefficients

print("Updated network_grid and projected_network with chunk=2.")
print("Free VRAM:", f"{torch.cuda.mem_get_info()[0]/1e9:.2f} GB / {torch.cuda.mem_get_info()[1]/1e9:.2f} GB")


Updated network_grid and projected_network with chunk=2.
Free VRAM: 42.81 GB / 85.09 GB


In [14]:
import time
import numpy as np
import torch

def torch_dct1(t):
    N, M = t.shape[-2:]
    rev_x = t[..., :, 1:-1].flip(-1)
    y1 = torch.cat([t, rev_x], dim=-1)
    rev_y = y1[..., 1:-1, :].flip(-2)
    y2 = torch.cat([y1, rev_y], dim=-2)
    return torch.fft.rfft2(y2).real[..., :N, :M]

def torch_coeff(t):
    scale = (t.shape[-1] - 1)**2
    res = torch_dct1(t) / scale
    res[..., (0, -1), :] *= 0.5
    res[..., :, (0, -1)] *= 0.5
    return res

def torch_values(t):
    w = t.clone()
    w[..., 1:-1, :] *= 0.5
    w[..., :, 1:-1] *= 0.5
    return torch_dct1(w)

def torch_convection(field, h):
    deriv_x = torch.zeros_like(field)
    deriv_x[1:-1] = (field[2:] - field[:-2]) / (2 * h)
    deriv_y = torch.zeros_like(field)
    deriv_y[:, 1:-1] = (field[:, 2:] - field[:, :-2]) / (2 * h)
    return -field * (deriv_x + deriv_y)

def torch_step(current, field, f0, f1, weight, h):
    decay, a, right = weight
    g0 = torch_coeff(torch_convection(field, h))
    predicted = decay * current + a * (g0 + f0) + right * (f1 - f0)
    predictor = torch_values(predicted)
    g1 = torch_coeff(torch_convection(predictor, h))
    result = predicted + right * (g1 - g0)
    return result, torch_values(result)

def torch_spatial_interpolate(field, nodes):
    # Bilinear interpolation from (Nx, Ny) to (nodes, nodes)
    device = field.device
    theta = torch.linspace(0, field.shape[0] - 1, nodes, dtype=torch.float64, device=device)
    index = theta.floor().long().clamp(0, field.shape[0] - 2)
    fraction = theta - index
    x = (1 - fraction[:, None]) * field[index] + fraction[:, None] * field[index + 1]
    return (1 - fraction[None, :]) * x[:, index] + fraction[None, :] * x[:, index + 1]

def torch_point_values(field, points, cfg):
    N = field.shape[0]
    device = field.device
    tx = ((points[:, 0] - cfg.x_min) / (cfg.x_max - cfg.x_min) * (N - 1)).clamp(0, N - 1)
    ty = ((points[:, 1] - cfg.y_min) / (cfg.y_max - cfg.y_min) * (N - 1)).clamp(0, N - 1)
    ix = tx.floor().long().clamp(0, N - 2)
    iy = ty.floor().long().clamp(0, N - 2)
    ax = tx - ix
    ay = ty - iy
    return ((1 - ax) * (1 - ay) * field[ix, iy]
            + ax * (1 - ay) * field[ix + 1, iy]
            + (1 - ax) * ay * field[ix, iy + 1]
            + ax * ay * field[ix + 1, iy + 1])

def torch_quadratic(a, b, c, fraction, first=False):
    if first:
        return 0.5 * (fraction - 1) * (fraction - 2) * a + fraction * (2 - fraction) * b + 0.5 * fraction * (fraction - 1) * c
    return 0.5 * fraction * (fraction - 1) * a + (1 - fraction ** 2) * b + 0.5 * fraction * (fraction + 1) * c

def torch_etd_weights(nodes, h, nu, dt, device="cuda"):
    k = torch.arange(nodes, dtype=torch.float64, device=device)
    one = 4 * nu / h ** 2 * torch.sin(torch.pi * k / (2 * (nodes - 1))) ** 2
    lam = one[:, None] + one[None, :]
    z = lam * dt
    safe = torch.where(z == 0, 1.0, z)
    a = torch.where(z == 0, dt, dt * -torch.expm1(-z) / safe)
    series = 0.5 - z / 6 + z ** 2 / 24 - z ** 3 / 120 + z ** 4 / 720 - z ** 5 / 5040
    phi2 = torch.where(z.abs() < 0.01, series, (z + torch.expm1(-z)) / safe ** 2)
    return torch.exp(-z), a, dt * phi2

def solve_points_gpu(record, cfg, nodes, factor, points, device="cuda"):
    tick = time.perf_counter()
    rate = record["forcing_u"] + cfg.jump_intensity * record["forcing_vg"]
    rate = rate.to(device=device, dtype=torch.float64)
    ref_u0 = record["reference"][:, :, 0].to(device=device, dtype=torch.float64)

    field = torch_spatial_interpolate(ref_u0, nodes)
    current = torch_coeff(field)
    steps = (rate.shape[-1] - 1) * factor
    dt = (cfg.t_max - cfg.t_min) / steps
    h = (cfg.x_max - cfg.x_min) / (nodes - 1)
    weight = torch_etd_weights(nodes, h, cfg.nu_u, dt, device=device)

    pts = points.to(device=device, dtype=torch.float64)
    theta = ((pts[:, 2] - cfg.t_min) / (cfg.t_max - cfg.t_min) * steps).clamp(0, steps)
    index = theta.floor().long().clamp(0, steps - 1)
    fraction = theta - index

    order = torch.argsort(index, stable=True)
    sorted_index = index[order]
    output = torch.full((len(points), 1), float("nan"), dtype=torch.float64, device=device)

    left = torch_coeff(torch_spatial_interpolate(rate[:, :, 0], nodes))
    max_abs = float(field.abs().max().item())

    for k in range(rate.shape[-1] - 1):
        right = torch_coeff(torch_spatial_interpolate(rate[:, :, k + 1], nodes))
        delta = right - left
        before = None
        for j in range(factor):
            number = k * factor + j
            current, next_field = torch_step(
                current, field,
                left + (j / factor) * delta,
                left + ((j + 1) / factor) * delta,
                weight, h
            )
            if j >= 1:
                begin = torch.searchsorted(sorted_index, number)
                end = torch.searchsorted(sorted_index, number + 1)
                selected = order[begin:end]
                if len(selected):
                    output[selected, 0] = torch_quadratic(
                        torch_point_values(before, pts[selected], cfg),
                        torch_point_values(field, pts[selected], cfg),
                        torch_point_values(next_field, pts[selected], cfg),
                        fraction[selected]
                    )
                if j == 1:
                    begin = torch.searchsorted(sorted_index, number - 1)
                    end = torch.searchsorted(sorted_index, number)
                    selected = order[begin:end]
                    if len(selected):
                        output[selected, 0] = torch_quadratic(
                            torch_point_values(before, pts[selected], cfg),
                            torch_point_values(field, pts[selected], cfg),
                            torch_point_values(next_field, pts[selected], cfg),
                            fraction[selected],
                            first=True
                        )
            before, field = field, next_field
        max_abs = max(max_abs, float(field.abs().max().item()))
        left = right

    elapsed = time.perf_counter() - tick
    return output, {"elapsed": elapsed, "max_abs": max_abs}


# ==============================================================================
# PRIORITY 1: RESERVED CONFIRMATION CAMPAIGN (15 CASES)
# Protocol: Frozen width-128, 1,200-step Causal3 solver (20 iterations)
# 5 unseen seeds (s in {3, 4, 5, 6, 7}) x 3 regimes (base, viscosity02, jump2)
# Acceptance Criteria: Every case < 10.0%, 15-case mean < 8.0%, zero divergence
# ==============================================================================

import os, sys, math, time, dataclasses, hashlib, json, subprocess
from fractions import Fraction
import numpy as np
import scipy
import torch
import torch.nn as nn
import torch.nn.functional as F

print("=" * 90)
print("     PRIORITY 1: RESERVED CONFIRMATION CAMPAIGN (15 CASES)     ")
print(f"Device: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM)")
print("Protocol: Frozen Width-128, 1,200 Steps/Iter, 20 Picard Iterations, Causal3")
print("Acceptance Criteria: Max case < 10.0%, Mean < 8.0%, 100% Stability")
print("=" * 90)

SEEDS = [3, 4, 5, 6, 7]
REGIMES = [
    {"name": "base", "nu_u": 0.01, "jump_intensity": 0.5},
    {"name": "viscosity02", "nu_u": 0.02, "jump_intensity": 0.5},
    {"name": "jump2", "nu_u": 0.01, "jump_intensity": 2.0},
]

WIDTH = 128
BATCH_SIZE = 4096
SELECTION_DRAWS = 8192
PICARD_ITERATIONS = 20
STEPS_PER_ITER = 1200
LR = 0.001
MODES = 128
HEAT_MODES = 128
POLYNOMIAL_NODES = 257
TIME_NODES = 512
DAMPING = 0.5
WINDOW_INTERVALS = 2
LOCAL_ITERATIONS = 3

os.makedirs("/content/benchmark/priority1_15cases", exist_ok=True)

# Spatial operator basis
nodes = POLYNOMIAL_NODES
full_basis = np.cos(np.pi * np.linspace(0, 1, nodes)[:, None] * np.arange(HEAT_MODES))
basis_128 = full_basis[:, :MODES]
derivative_128 = derivative_matrix(MODES, nodes, 2.0)

results_15 = {}
case_index = 0

for s in SEEDS:
    # 1. Base Realization
    rec_base, cfg_base = build_frozen_realisation(s)

    for reg in REGIMES:
        case_index += 1
        reg_name = reg["name"]
        case_name = f"s{s}_{reg_name}"

        print("\n" + "=" * 80)
        print(f"[{case_index:2d}/15] STARTING RESERVED CASE: {case_name}")
        print(f"       Seed={s} | Regime={reg_name} (nu_u={reg['nu_u']}, jump_intensity={reg['jump_intensity']})")
        print("=" * 80)
        t_case_start = time.time()

        # Build case config
        cfg = dataclasses.replace(cfg_base, nu_u=reg["nu_u"], jump_intensity=reg["jump_intensity"])

        # Heat Lift for this regime
        lift = HeatLift(rec_base, cfg, modes=HEAT_MODES)
        q_grid = full_basis @ lift.knots.cpu().numpy() @ full_basis.T
        times = torch.linspace(cfg.t_min, cfg.t_max, TIME_NODES, dtype=torch.float64, device="cuda")

        # Test points (historical offset 290000 + s)
        gen_test = torch.Generator(device="cuda").manual_seed(290000 + s)
        test_coll = sample_points(cfg, 32768, gen_test, "cuda")
        q_test = lift.evaluate(test_coll)

        # Continuum ETD2 Ground Truth Reference (nodes=2049, factor=4)
        print(f"Solving Continuum ETD2 Reference (2049x2049, 2045 steps) on GPU...")
        t0 = time.time()
        targets, info = solve_points_gpu(rec_base, cfg, nodes=2049, factor=4, points=test_coll, device="cuda")
        t_ref = time.time() - t0
        target_rms = targets.square().mean().sqrt().item()
        q_err = (q_test - targets).square().mean().sqrt().item() / target_rms
        print(f"ETD2 Ground Truth computed in {t_ref:.2f}s | Target RMS: {target_rms:.6f} | Heat Lift baseline err: {q_err:.4%}")

        # Model Initialization (Width 128, 4 layers, Float64)
        torch.manual_seed(80000 + s)
        base_pinn = ScalarPINN(hidden_layers=4, hidden_dim=WIDTH, num_fourier_features=64, fourier_scale=3.0, device="cuda")
        system = ConditionedCorrection(base_pinn, cfg, hard_boundary=True).to(device="cuda", dtype=torch.float64)

        # Amplitude bound
        amplitude = (1 + Fraction(float(rec_base["reference"][:, :, 0].abs().max()))
            + (Fraction(cfg.t_max) - Fraction(cfg.t_min))
            * (Fraction(float(rec_base["forcing_u"].abs().max()))
               + abs(Fraction(cfg.jump_intensity)) * Fraction(float(rec_base["forcing_vg"].abs().max()))))

        gen_sel = torch.Generator(device="cuda").manual_seed(360000 + s)
        selection_points = sample_points(cfg, SELECTION_DRAWS, gen_sel, "cuda")
        gen_train = torch.Generator(device="cuda").manual_seed(350000 + s)

        history_case = []

        # 20 Picard Iterations x 1200 steps
        for iteration in range(1, PICARD_ITERATIONS + 1):
            t_iter = time.time()

            with torch.no_grad():
                coef = projected_network(system, cfg, times, nodes=POLYNOMIAL_NODES, modes=MODES)
                target_coef = causal_picard_target(
                    coef, q_grid, basis_128, derivative_128, cfg,
                    damping=DAMPING, window_intervals=WINDOW_INTERVALS, local_iterations=LOCAL_ITERATIONS
                )
                target_grid = regression_grid(target_coef, modes=MODES, nodes=POLYNOMIAL_NODES, device="cuda")
                selected_target = interpolate_points(target_grid, selection_points, cfg)

                w_sel = system(selection_points[:, :1], selection_points[:, 1:2], selection_points[:, 2:3])
                best_loss = (w_sel - selected_target).square().mean().item()
                best_step = 0
                best_state = {k: v.detach().clone() for k, v in system.state_dict().items()}

            optimizer = torch.optim.Adam(system.parameters(), lr=LR)

            for step in range(1, STEPS_PER_ITER + 1):
                points = sample_points(cfg, BATCH_SIZE, gen_train, "cuda")
                batch_targets = interpolate_points(target_grid, points, cfg)

                optimizer.zero_grad(set_to_none=True)
                w_pred = system(points[:, :1], points[:, 1:2], points[:, 2:3])
                loss = (w_pred - batch_targets).square().mean()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(system.parameters(), 5.0, error_if_nonfinite=True)
                optimizer.step()
                project_parameters(system, WIDTH, amplitude)

                if step % 100 == 0:
                    with torch.no_grad():
                        w_sel = system(selection_points[:, :1], selection_points[:, 1:2], selection_points[:, 2:3])
                        measured = (w_sel - selected_target).square().mean().item()
                        if measured < best_loss:
                            best_loss = measured
                            best_step = step
                            best_state = {k: v.detach().clone() for k, v in system.state_dict().items()}

            system.load_state_dict(best_state)
            elapsed = time.time() - t_iter

            if iteration in (1, 5, 10, 15, 20):
                with torch.no_grad():
                    w_test = system(test_coll[:, :1], test_coll[:, 1:2], test_coll[:, 2:3])
                    pred = w_test + q_test
                    err = pred - targets
                    rel_rms = math.sqrt(err.square().mean().item()) / target_rms
                history_case.append({"iteration": iteration, "rel_rms": rel_rms, "best_step": best_step, "elapsed": elapsed})
                print(f"[{case_name} | ITER {iteration:2d}/20] ({elapsed:.1f}s) Best step: {best_step:4d} | Rel RMS: {rel_rms:.4%}", flush=True)

        final_rel_rms = history_case[-1]["rel_rms"]
        case_total_time = time.time() - t_case_start
        passed = final_rel_rms < 0.10

        results_15[case_name] = {
            "seed": s,
            "regime": reg_name,
            "nu_u": reg["nu_u"],
            "jump_intensity": reg["jump_intensity"],
            "target_rms": target_rms,
            "heat_lift_err": q_err,
            "final_rel_rms": final_rel_rms,
            "history": history_case,
            "passed_10pct_gate": passed,
            "elapsed_seconds": case_total_time
        }

        torch.save({"model": system.state_dict(), "history": history_case, "final_rel_rms": final_rel_rms},
                   f"/content/benchmark/priority1_15cases/{case_name}.pt")
        with open("/content/benchmark/priority1_15cases/summary.json", "w") as f:
            json.dump(results_15, f, indent=2)

        print(f">>> {case_name} COMPLETE in {case_total_time:.1f}s | Final Rel RMS: {final_rel_rms:.4%} | Status: {'PASS' if passed else 'FAIL'} <<<\n", flush=True)

print("\n" + "=" * 90)
print("             PRIORITY 1: 15-CASE RESERVED CONFIRMATION FINAL AUDIT             ")
print("=" * 90)
print(f"{'Case Name':<20} | {'Regime':<12} | {'Target RMS':<12} | {'Lift Err':<12} | {'Final Rel RMS':<14} | {'Status'}")
print("-" * 90)
for name, r in results_15.items():
    status = "PASS (<10%)" if r["passed_10pct_gate"] else "FAIL"
    print(f"{name:<20} | {r['regime']:<12} | {r['target_rms']:<12.6f} | {r['heat_lift_err']:<12.4%} | {r['final_rel_rms']:<14.4%} | {status}")

mean_err = sum(r["final_rel_rms"] for r in results_15.values()) / len(results_15)
max_err = max(r["final_rel_rms"] for r in results_15.values())
all_passed = all(r["passed_10pct_gate"] for r in results_15.values()) and mean_err < 0.08

print("=" * 90)
print(f"15-Case Ensemble Mean Rel RMS: {mean_err:.4%} (Threshold: < 8.0000%)")
print(f"15-Case Maximum Rel RMS:       {max_err:.4%} (Threshold: < 10.0000%)")
print(f"Overall Campaign Status:       {'ALL GATES PASSED (CONFIRMED)' if all_passed else 'FAILED'}")
print("=" * 90)


     PRIORITY 1: RESERVED CONFIRMATION CAMPAIGN (15 CASES)     
Device: NVIDIA A100-SXM4-80GB (85.1 GB VRAM)
Protocol: Frozen Width-128, 1,200 Steps/Iter, 20 Picard Iterations, Causal3
Acceptance Criteria: Max case < 10.0%, Mean < 8.0%, 100% Stability

[ 1/15] STARTING RESERVED CASE: s3_base
       Seed=3 | Regime=base (nu_u=0.01, jump_intensity=0.5)
Solving Continuum ETD2 Reference (2049x2049, 2045 steps) on GPU...
ETD2 Ground Truth computed in 19.03s | Target RMS: 0.248538 | Heat Lift baseline err: 45.4764%
[s3_base | ITER  1/20] (14.3s) Best step: 1200 | Rel RMS: 16.1257%
[s3_base | ITER  5/20] (14.2s) Best step: 1000 | Rel RMS: 8.3678%
[s3_base | ITER 10/20] (16.7s) Best step:  800 | Rel RMS: 6.5216%
[s3_base | ITER 15/20] (16.6s) Best step:    0 | Rel RMS: 6.1631%
[s3_base | ITER 20/20] (14.4s) Best step:    0 | Rel RMS: 6.1631%
>>> s3_base COMPLETE in 330.7s | Final Rel RMS: 6.1631% | Status: PASS <<<


[ 2/15] STARTING RESERVED CASE: s3_viscosity02
       Seed=3 | Regime=viscosi

In [15]:
# ==============================================================================
# PRIORITY 2: TRAINING VARIABILITY & FAIR COST COMPARISONS (GOOGLE COLAB A100)
# ==============================================================================
import os, sys, math, time, dataclasses, hashlib, json
from dataclasses import dataclass
from fractions import Fraction
import numpy as np
import scipy
from scipy.linalg import solve_banded
from scipy.fft import dctn
import torch
import torch.nn as nn
import torch.nn.functional as F

print("=" * 90)
print("     PRIORITY 2: TRAINING VARIABILITY & FAIR COST COMPARISONS     ")
print(f"Device: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM)")
print("=" * 90)

@dataclass
class ConfigBurgersVG2D:
    x_min: float = -1.0
    x_max: float =  1.0
    y_min: float = -1.0
    y_max: float =  1.0
    t_min: float =  0.0
    t_max: float =  0.5
    Nx: int = 64
    Ny: int = 64
    Nt: int = 512
    nu_u: float = 1.0e-2
    sigma_u: float = 0.05
    noise_type_u: str = "white"
    hurst_u: float = 0.7
    sigma_vg: float = 0.10
    nu_vg: float = 0.20
    theta_vg: float = -0.03
    jump_intensity: float = 0.5
    ic_amplitude: float = 0.6
    burn_t: float = 1.0
    hidden_layers: int = 4
    hidden_dim: int = 128
    num_fourier_features: int = 64
    fourier_scale: float = 3.0

    @property
    def dx(self): return (self.x_max - self.x_min) / (self.Nx - 1)
    @property
    def dy(self): return (self.y_max - self.y_min) / (self.Ny - 1)
    @property
    def dt(self): return (self.t_max - self.t_min) / (self.Nt - 1)

def generate_vg_jump_field(Nx, Ny, Nt, dt, sigma_vg=0.10, nu_vg=0.20, theta_vg=-0.03, rng=None):
    if rng is None:
        rng = np.random.default_rng()
    out = np.zeros((Nx, Ny, Nt), dtype=np.float32)
    for k in range(Nt):
        G = rng.gamma(shape=dt / nu_vg, scale=nu_vg, size=(Nx, Ny))
        Z = rng.normal(0.0, 1.0, size=(Nx, Ny))
        out[:, :, k] = (theta_vg * G + sigma_vg * np.sqrt(np.maximum(G, 1e-12)) * Z)
    return out

def _tridiag_neumann(N, r):
    half_r = 0.5 * r
    diag = np.full(N, 1 + 2 * half_r); off = np.full(N - 1, -half_r)
    diag[0] = 1 + half_r; diag[-1] = 1 + half_r
    lhs = np.zeros((3, N))
    lhs[0, 1:] = off; lhs[1, :] = diag; lhs[2, :-1] = off
    d_rhs = np.full(N, 1 - 2 * half_r)
    d_rhs[0] = 1 - half_r; d_rhs[-1] = 1 - half_r
    o_rhs = half_r
    def rhs_mult(u):
        v = d_rhs * u
        v[:-1] += o_rhs * u[1:]; v[1:] += o_rhs * u[:-1]
        return v
    return lhs, rhs_mult

def _adi_step(u, D, dx, dy, dt, forcing):
    Nx, Ny = u.shape
    rx = D * dt / dx ** 2; ry = D * dt / dy ** 2
    lhs_x, rhs_x = _tridiag_neumann(Nx, rx)
    lhs_y, rhs_y = _tridiag_neumann(Ny, ry)
    u_half = np.empty_like(u)
    for j in range(Ny):
        col = u[:, j]
        if 0 < j < Ny - 1:
            yyc = ry * 0.5 * (u[:, j+1] - 2*u[:, j] + u[:, j-1])
        elif j == 0:
            yyc = ry * 0.5 * (u[:, j+1] - u[:, j])
        else:
            yyc = ry * 0.5 * (u[:, j-1] - u[:, j])
        rhs_vec = col + yyc + 0.5 * dt * forcing[:, j]
        u_half[:, j] = solve_banded((1, 1), lhs_x, rhs_vec)
    u_new = np.empty_like(u)
    for i in range(Nx):
        row = u_half[i, :]
        if 0 < i < Nx - 1:
            xxc = rx * 0.5 * (u_half[i+1, :] - 2*u_half[i, :] + u_half[i-1, :])
        elif i == 0:
            xxc = rx * 0.5 * (u_half[i+1, :] - u_half[i, :])
        else:
            xxc = rx * 0.5 * (u_half[i-1, :] - u_half[i, :])
        rhs_vec = row + xxc + 0.5 * dt * forcing[i, :]
        u_new[i, :] = solve_banded((1, 1), lhs_y, rhs_vec)
    return u_new

def _upwind_advection(u, dx, dy):
    pos = (u >= 0)
    bwd_x = np.zeros_like(u); fwd_x = np.zeros_like(u)
    bwd_x[1:, :] = (u[1:, :] - u[:-1, :]) / dx
    fwd_x[:-1, :] = (u[1:, :] - u[:-1, :]) / dx
    u_x = np.where(pos, bwd_x, fwd_x)
    bwd_y = np.zeros_like(u); fwd_y = np.zeros_like(u)
    bwd_y[:, 1:] = (u[:, 1:] - u[:, :-1]) / dy
    fwd_y[:, :-1] = (u[:, 1:] - u[:, :-1]) / dy
    u_y = np.where(pos, bwd_y, fwd_y)
    return u * (u_x + u_y)

def _ic_u(X, Y, cfg, rng):
    base = cfg.ic_amplitude * np.sin(np.pi * X) * np.sin(np.pi * Y)
    perturb = 0.05 * rng.standard_normal(X.shape)
    return base + perturb

def build_frozen_realisation(seed, device="cuda"):
    cfg = ConfigBurgersVG2D(t_min=0.0, t_max=0.5, Nt=512)
    burn_t = 1.0
    dt0 = cfg.dt
    n_burn = int(round(burn_t / dt0))
    Nt_gen = cfg.Nt + n_burn
    t_max_gen = (Nt_gen - 1) * dt0
    gen = dataclasses.replace(cfg, t_min=0.0, Nt=Nt_gen, t_max=t_max_gen)

    rng = np.random.default_rng(seed)
    raw = rng.standard_normal((gen.Nx, gen.Ny, gen.Nt))
    noise = ((raw.astype(np.float32) * np.float32(gen.sigma_u)) / np.float32(np.sqrt(gen.dx * gen.dy * gen.dt)))

    rng_vg = np.random.default_rng(seed + 10000)
    vg_inc = generate_vg_jump_field(
        gen.Nx, gen.Ny, gen.Nt, gen.dt,
        sigma_vg=gen.sigma_vg, nu_vg=gen.nu_vg, theta_vg=gen.theta_vg, rng=rng_vg)

    vg_rate = vg_inc / gen.dt
    x_grid = np.linspace(gen.x_min, gen.x_max, gen.Nx)
    y_grid = np.linspace(gen.y_min, gen.y_max, gen.Ny)
    t_grid = np.linspace(gen.t_min, gen.t_max, gen.Nt)
    X, Yg = np.meshgrid(x_grid, y_grid, indexing="ij")
    U = np.zeros((gen.Nx, gen.Ny, gen.Nt), dtype=np.float32)
    U[:, :, 0] = _ic_u(X, Yg, gen, rng)

    for n in range(gen.Nt - 1):
        u_n = U[:, :, n].astype(np.float64)
        advect = _upwind_advection(u_n, gen.dx, gen.dy)
        forcing = -advect + noise[:, :, n] + gen.jump_intensity * vg_rate[:, :, n]
        u_np1 = _adi_step(u_n, gen.nu_u, gen.dx, gen.dy, gen.dt, forcing)
        u_np1 = np.clip(u_np1, -3.0, 3.0)
        U[:, :, n + 1] = u_np1.astype(np.float32)

    noise = noise[:, :, n_burn:]
    vg_rate = vg_rate[:, :, n_burn:]
    U = U[:, :, n_burn:]
    t_grid = t_grid[n_burn:]

    record = {
        "reference": torch.tensor(U, dtype=torch.float64, device=device),
        "forcing_u": torch.tensor(noise, dtype=torch.float32, device=device).double(),
        "forcing_vg": torch.tensor(vg_rate, dtype=torch.float32, device=device).double(),
        "x_grid": torch.tensor(x_grid, dtype=torch.float64, device=device),
        "y_grid": torch.tensor(y_grid, dtype=torch.float64, device=device),
        "t_grid": torch.tensor(t_grid, dtype=torch.float64, device=device),
    }
    target_cfg = dataclasses.replace(cfg, t_min=float(t_grid[0]), t_max=float(t_grid[-1]))
    return record, target_cfg

class FourierFeatures(nn.Module):
    def __init__(self, num_features: int = 64, scale: float = 3.0, in_dim: int = 3, device="cuda"):
        super().__init__()
        B = scale * torch.randn(in_dim, num_features, dtype=torch.float64, device=device)
        self.register_buffer("B", B)

    def forward(self, x, y, t):
        v = torch.cat([x, y, t], dim=-1)
        proj = 2.0 * math.pi * v @ self.B
        return torch.cat([torch.sin(proj), torch.cos(proj)], dim=-1)

class ScalarPINN(nn.Module):
    def __init__(self, hidden_layers: int = 4, hidden_dim: int = 128,
                 num_fourier_features: int = 64, fourier_scale: float = 3.0, device="cuda"):
        super().__init__()
        self.fourier = FourierFeatures(num_fourier_features, fourier_scale, in_dim=3, device=device)
        in_dim = 2 * num_fourier_features
        layers = [nn.Linear(in_dim, hidden_dim, dtype=torch.float64, device=device), nn.Tanh()]
        for _ in range(hidden_layers - 1):
            layers += [nn.Linear(hidden_dim, hidden_dim, dtype=torch.float64, device=device), nn.Tanh()]
        head = nn.Linear(hidden_dim, 1, dtype=torch.float64, device=device)
        with torch.no_grad():
            head.weight.zero_()
            head.bias.zero_()
        layers.append(head)
        self.net = nn.Sequential(*layers)

    def forward(self, x, y, t):
        feat = self.fourier(x, y, t)
        return self.net(feat)

class ConditionedCorrection(nn.Module):
    def __init__(self, base: ScalarPINN, cfg, hard_boundary: bool = True):
        super().__init__()
        self.base = base
        self.cfg = cfg
        self.hard_boundary = hard_boundary

    def forward(self, x, y, t):
        if self.hard_boundary:
            x_b = torch.cos(math.pi * (x - self.cfg.x_min) / (self.cfg.x_max - self.cfg.x_min))
            y_b = torch.cos(math.pi * (y - self.cfg.y_min) / (self.cfg.y_max - self.cfg.y_min))
        else:
            x_b, y_b = x, y
        value = self.base(x_b, y_b, t)
        envelope = (t - self.cfg.t_min) / (self.cfg.t_max - self.cfg.t_min)
        return envelope * value

def project_parameters(model, width, amplitude):
    with torch.no_grad():
        for p in model.parameters():
            p.clamp_(-width, width)
        head = model.base.net[-1]
        total = head.weight.abs().sum() + head.bias.abs().sum()
        limit = float(amplitude - Fraction(5, 8))
        if total.item() > limit:
            factor = limit / total
            head.weight.mul_(factor)
            head.bias.mul_(factor)

def hat_coefficients(nodes, modes, length, device="cpu"):
    h = length / (nodes - 1)
    k = torch.arange(modes, dtype=torch.float64, device=device)
    omega = k * math.pi / length
    omega_safe = omega.clone()
    omega_safe[0] = 1
    scale = 2 * torch.sin(0.5 * omega * h).square() / (h * omega_safe.square())
    x = torch.arange(nodes, dtype=torch.float64, device=device) * h
    coefficients = 2 * scale[:, None] * torch.cos(omega[:, None] * x[None])
    coefficients[:, [0, -1]] *= 0.5
    coefficients[0] = h
    coefficients[0, [0, -1]] = 0.5 * h
    coefficients[1:] *= 2 / length
    coefficients[0] /= length
    return coefficients

def modal_weights(eigenvalues, dt):
    z = eigenvalues * dt
    safe = eigenvalues.clone()
    safe[safe == 0] = 1
    a = -torch.expm1(-z) / safe
    series = dt**2 * (0.5 - z/6 + z.square()/24 - z**3/120 + z**4/720 - z**5/5040)
    b = torch.where(z.abs() < 0.01, series, (dt - a) / safe)
    a = torch.where(eigenvalues == 0, torch.full_like(a, dt), a)
    return torch.exp(-z), a, b

class HeatLift:
    def __init__(self, record, cfg, modes=128):
        self.cfg = cfg
        self.modes = modes
        rate = record["forcing_u"] + cfg.jump_intensity * record["forcing_vg"]
        ax = hat_coefficients(rate.shape[0], modes, cfg.x_max - cfg.x_min, rate.device)
        ay = hat_coefficients(rate.shape[1], modes, cfg.y_max - cfg.y_min, rate.device)
        self.forcing = torch.einsum("ki,ijt,lj->tkl", ax, rate, ay).contiguous()
        self.initial = ax @ record["reference"][:, :, 0] @ ay.T
        k = torch.arange(modes, dtype=torch.float64, device=rate.device)
        self.wx = k * math.pi / (cfg.x_max - cfg.x_min)
        self.wy = k * math.pi / (cfg.y_max - cfg.y_min)
        self.lam = cfg.nu_u * (self.wx[:, None].square() + self.wy[None].square())
        self.dt = (cfg.t_max - cfg.t_min) / (len(self.forcing) - 1)
        decay, a, b = modal_weights(self.lam, self.dt)
        self.knots = torch.empty_like(self.forcing)
        self.knots[0] = self.initial
        for j in range(len(self.forcing) - 1):
            self.knots[j + 1] = decay * self.knots[j] + a * self.forcing[j] + (b / self.dt) * (self.forcing[j + 1] - self.forcing[j])

    def evaluate(self, points, chunk=4096):
        out = []
        for start in range(0, len(points), chunk):
            pts = points[start:start+chunk]
            index = ((pts[:, 2] - self.cfg.t_min) / self.dt).floor().long().clamp(0, len(self.forcing) - 2)
            elapsed = (pts[:, 2] - self.cfg.t_min - index.to(pts.dtype) * self.dt).view(-1, 1, 1)
            safe = self.lam.clone()
            safe[safe == 0] = 1
            z = self.lam * elapsed
            a = -torch.expm1(-z) / safe
            a = torch.where(self.lam == 0, elapsed, a)
            series = elapsed.square() * (0.5 - z/6 + z.square()/24 - z**3/120 + z**4/720 - z**5/5040)
            b = torch.where(z.abs() < 0.01, series, (elapsed - a) / safe)
            left, right = self.forcing[index], self.forcing[index + 1]
            coefficients = torch.exp(-z) * self.knots[index] + a * left + (b / self.dt) * (right - left)
            px = (pts[:, :1] - self.cfg.x_min) * self.wx
            py = (pts[:, 1:2] - self.cfg.y_min) * self.wy
            cx, cy = px.cos(), py.cos()
            q = torch.einsum("bi,bij,bj->b", cx, coefficients, cy).view(-1, 1)
            out.append(q)
        return torch.cat(out, dim=0)

def derivative_matrix(modes, square_modes, length):
    k = np.arange(modes)[:, None]
    m = np.arange(square_modes)[None, :]
    plus, minus = m + k, m - k
    def fraction(n):
        return np.divide(2 * (n % 2 != 0), n, out=np.zeros_like(n, dtype=float), where=n != 0)
    normal = np.where(k == 0, 1.0, 2.0)
    return -normal * m / (4 * length) * (fraction(plus) + fraction(minus))

def cosine_coefficients(values):
    assert values.shape[-2] == values.shape[-1]
    result = dctn(values, type=1, axes=(-2, -1), workers=4) / (values.shape[-1] - 1) ** 2
    result[..., (0, -1), :] *= 0.5
    result[..., :, (0, -1)] *= 0.5
    return result

def nonlinear_source(q, w, basis, derivative, chunk=32):
    modes = w.shape[-1]
    result = np.empty_like(w)
    for start in range(0, len(w), chunk):
        stop = min(start + chunk, len(w))
        values = q[start:stop] + basis @ w[start:stop] @ basis.T
        with np.errstate(over="ignore", invalid="ignore"):
            square = cosine_coefficients(values * values)
            result[start:stop] = (derivative @ square[:, :, :modes]
                                  + square[:, :modes, :] @ derivative.T)
    return result

def response_ode(source, initial, nu, lx, ly, dt):
    modes = source.shape[-1]
    k = np.arange(modes)
    lam = nu * ((np.pi * k[:, None] / lx) ** 2 + (np.pi * k[None, :] / ly) ** 2)
    z = lam * dt
    decay = np.exp(-z)
    safe = np.where(z == 0, 1, z)
    a = -np.expm1(-z) / safe
    right = (z + np.expm1(-z)) / safe ** 2
    small = np.abs(z) < 0.01
    right = np.where(small, 0.5 - z / 6 + z ** 2 / 24 - z ** 3 / 120 + z ** 4 / 720 - z ** 5 / 5040, right)
    a = np.where(z == 0, 1, a)
    left, right = dt * (a - right), dt * right
    output = np.empty_like(source)
    output[0] = initial
    with np.errstate(over="ignore", invalid="ignore"):
        for j in range(len(source) - 1):
            output[j + 1] = decay * output[j] + left * source[j] + right * source[j + 1]
    return output

def causal_picard_target(neural, q, basis, derivative, cfg, damping=0.5, window_intervals=2, local_iterations=3):
    count, modes = neural.shape[:2]
    dt = (cfg.t_max - cfg.t_min) / (count - 1)
    initial = np.zeros((modes, modes))
    lam = cfg.nu_u * ((np.pi * np.arange(modes)[:, None] / (cfg.x_max - cfg.x_min)) ** 2
                      + (np.pi * np.arange(modes)[None, :] / (cfg.y_max - cfg.y_min)) ** 2)
    result = np.empty_like(neural)
    for start in range(0, count - 1, window_intervals):
        stop = min(start + window_intervals, count - 1)
        decay = np.exp(-np.arange(stop - start + 1)[:, None, None] * dt * lam)
        waveform = neural[start:stop + 1] + decay * (initial - neural[start])
        for _ in range(local_iterations):
            source = nonlinear_source(q[start:stop + 1], waveform, basis, derivative)
            h = response_ode(-source, initial, cfg.nu_u, cfg.x_max - cfg.x_min, cfg.y_max - cfg.y_min, dt)
            waveform = (1 - damping) * waveform + damping * h
        waveform[0] = initial
        result[start:stop + 1] = waveform
        initial = waveform[-1].copy()
    return result

def network_grid(system, cfg, times, nodes=257, chunk=2):
    x = torch.linspace(cfg.x_min, cfg.x_max, nodes, dtype=times.dtype, device=times.device)
    y = torch.linspace(cfg.y_min, cfg.y_max, nodes, dtype=times.dtype, device=times.device)
    X, Y = torch.meshgrid(x, y, indexing="ij")
    output = times.new_empty(len(times), nodes, nodes)
    with torch.no_grad():
        for start in range(0, len(times), chunk):
            count = min(chunk, len(times) - start)
            xx = X[None].expand(count, -1, -1).reshape(-1, 1)
            yy = Y[None].expand(count, -1, -1).reshape(-1, 1)
            tt = times[start:start+count, None, None].expand(-1, nodes, nodes).reshape(-1, 1)
            output[start:start+count] = system(xx, yy, tt).view(count, nodes, nodes)
    return output

def projected_network(system, cfg, times, nodes=257, modes=128, chunk=2):
    values = network_grid(system, cfg, times, nodes=nodes, chunk=chunk).cpu().numpy()
    coefficients = cosine_coefficients(values)[:, :modes, :modes].copy()
    return coefficients

def regression_grid(coefficients, modes=128, nodes=257, device="cuda"):
    coef = torch.from_numpy(coefficients).to(device)
    z = torch.linspace(0, 1, nodes, dtype=torch.float64, device=device)
    k = torch.arange(modes, dtype=torch.float64, device=device)
    basis = (torch.pi * z[:, None] * k[None]).cos()
    output = coef.new_empty(len(coef), nodes, nodes)
    for start in range(0, len(coef), 16):
        count = min(16, len(coef) - start)
        output[start:start+count] = torch.einsum("xi,bij,yj->bxy", basis, coef[start:start+count], basis)
    return output.permute(1, 2, 0).contiguous()

def interpolate_points(field, points, cfg):
    lengths = field.new_tensor([cfg.x_max - cfg.x_min, cfg.y_max - cfg.y_min, cfg.t_max - cfg.t_min])
    lower = field.new_tensor([cfg.x_min, cfg.y_min, cfg.t_min])
    sizes = field.new_tensor(field.shape) - 1
    q = ((points - lower) / lengths * sizes).clamp(min=0)
    q = torch.minimum(q, sizes)
    left = q.floor().long()
    right = torch.minimum(left + 1, sizes.long())
    weight = q - left
    result = points.new_zeros(len(points))
    for a in (0, 1):
        for b in (0, 1):
            for c in (0, 1):
                indices = [right[:, i] if side else left[:, i] for i, side in enumerate((a, b, c))]
                factor = torch.ones_like(result)
                for i, side in enumerate((a, b, c)):
                    factor = factor * (weight[:, i] if side else 1 - weight[:, i])
                result += factor * field[tuple(indices)]
    return result.view(-1, 1)

def sample_points(cfg, count, generator, device="cuda"):
    lo = torch.tensor([cfg.x_min, cfg.y_min, cfg.t_min], dtype=torch.float64, device=device)
    length = lo.new_tensor([cfg.x_max - cfg.x_min, cfg.y_max - cfg.y_min, cfg.t_max - cfg.t_min])
    return lo + length * torch.rand(count, 3, generator=generator, device=device, dtype=torch.float64)

# ==============================================================================
# BENCHMARK SETUP
# ==============================================================================
os.makedirs("/content/benchmark/priority2", exist_ok=True)

# 1. Base Realization for Driver 0
rec0, cfg0 = build_frozen_realisation(0)
MODES = 128
HEAT_MODES = 128
NODES = 257
TIME_NODES = 512
WIDTH = 128
BATCH_SIZE = 4096
SELECTION_DRAWS = 8192
LR = 0.001
DAMPING = 0.5
WINDOW_INTERVALS = 2

lift0 = HeatLift(rec0, cfg0, modes=HEAT_MODES)
full_basis = np.cos(np.pi * np.linspace(0, 1, NODES)[:, None] * np.arange(HEAT_MODES))
q_grid0 = full_basis @ lift0.knots.cpu().numpy() @ full_basis.T
basis_128 = full_basis[:, :MODES]
derivative_128 = derivative_matrix(MODES, NODES, cfg0.x_max - cfg0.x_min)
times0 = torch.linspace(cfg0.t_min, cfg0.t_max, TIME_NODES, dtype=torch.float64, device="cuda")

# Test points & audited targets
if not os.path.exists("/content/benchmark/test.pt"):
    os.system("curl -s -o /content/benchmark/test.pt https://litter.catbox.moe/lxq3vs.pt")
if not os.path.exists("/content/benchmark/n2049_f4_values.pt"):
    os.system("curl -s -o /content/benchmark/n2049_f4_values.pt https://litter.catbox.moe/32sy5y.pt")

test_arc = torch.load("/content/benchmark/test.pt", map_location="cuda", weights_only=True)
test_coll = test_arc["coll"]
q_test = test_arc["q"]
audited_targets = torch.load("/content/benchmark/n2049_f4_values.pt", map_location="cuda", weights_only=True)["all"][:len(test_coll)]
target_rms = audited_targets.square().mean().sqrt().item()

amplitude0 = (1 + Fraction(float(rec0["reference"][:, :, 0].abs().max()))
    + (Fraction(cfg0.t_max) - Fraction(cfg0.t_min))
    * (Fraction(float(rec0["forcing_u"].abs().max()))
       + abs(Fraction(cfg0.jump_intensity)) * Fraction(float(rec0["forcing_vg"].abs().max()))))

num_windows = (TIME_NODES - 1) // WINDOW_INTERVALS

def execute_run(tag, init_seed, local_iterations, picard_iterations, steps_per_iter):
    print(f"\n{'='*80}\n[START] {tag}\nInitSeed={init_seed} | K_local={local_iterations} | Iters={picard_iterations} | Steps/Iter={steps_per_iter}\n{'='*80}", flush=True)
    t_start = time.time()

    torch.manual_seed(init_seed)
    base_pinn = ScalarPINN(hidden_layers=4, hidden_dim=WIDTH, num_fourier_features=64, fourier_scale=3.0, device="cuda")
    system = ConditionedCorrection(base_pinn, cfg0, hard_boundary=True).to(device="cuda", dtype=torch.float64)

    gen_sel = torch.Generator(device="cuda").manual_seed(360000)
    selection_points = sample_points(cfg0, SELECTION_DRAWS, gen_sel, "cuda")
    gen_train = torch.Generator(device="cuda").manual_seed(350000)

    history = []
    total_evals = 0

    for iteration in range(1, picard_iterations + 1):
        t_iter = time.time()
        with torch.no_grad():
            coef = projected_network(system, cfg0, times0, nodes=NODES, modes=MODES, chunk=2)
            target_coef = causal_picard_target(
                coef, q_grid0, basis_128, derivative_128, cfg0,
                damping=DAMPING, window_intervals=WINDOW_INTERVALS, local_iterations=local_iterations
            )
            total_evals += num_windows * local_iterations
            target_grid = regression_grid(target_coef, modes=MODES, nodes=NODES, device="cuda")
            selected_target = interpolate_points(target_grid, selection_points, cfg0)
            w_sel = system(selection_points[:, :1], selection_points[:, 1:2], selection_points[:, 2:3])
            best_loss = (w_sel - selected_target).square().mean().item()
            best_step = 0
            best_state = {k: v.detach().clone() for k, v in system.state_dict().items()}

        optimizer = torch.optim.Adam(system.parameters(), lr=LR)
        for step in range(1, steps_per_iter + 1):
            points = sample_points(cfg0, BATCH_SIZE, gen_train, "cuda")
            targets = interpolate_points(target_grid, points, cfg0)
            optimizer.zero_grad(set_to_none=True)
            w_pred = system(points[:, :1], points[:, 1:2], points[:, 2:3])
            loss = (w_pred - targets).square().mean()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(system.parameters(), 5.0, error_if_nonfinite=True)
            optimizer.step()
            project_parameters(system, WIDTH, amplitude0)

            if step % 100 == 0:
                with torch.no_grad():
                    w_sel = system(selection_points[:, :1], selection_points[:, 1:2], selection_points[:, 2:3])
                    measured = (w_sel - selected_target).square().mean().item()
                    if measured < best_loss:
                        best_loss = measured
                        best_step = step
                        best_state = {k: v.detach().clone() for k, v in system.state_dict().items()}

        system.load_state_dict(best_state)
        elapsed = time.time() - t_iter
        cum_time = time.time() - t_start

        if iteration in (1, 5, 10, 15, 20, picard_iterations):
            with torch.no_grad():
                w_test = system(test_coll[:, :1], test_coll[:, 1:2], test_coll[:, 2:3])
                pred = w_test + q_test
                err = pred - audited_targets
                rel_rms = math.sqrt(err.square().mean().item()) / target_rms
            history.append({
                "iteration": iteration, "rel_rms": rel_rms, "best_step": best_step,
                "cum_evals": total_evals, "cum_time": cum_time, "iter_time": elapsed
            })
            print(f"[{tag} | ITER {iteration:2d}/{picard_iterations}] ({cum_time:5.1f}s, evals: {total_evals:5d}) Best step: {best_step:4d} | Rel RMS: {rel_rms:.4%}", flush=True)

    total_time = time.time() - t_start
    final_rel_rms = history[-1]["rel_rms"]
    print(f">>> {tag} COMPLETE in {total_time:.1f}s | Final Rel RMS: {final_rel_rms:.4%} <<<\n", flush=True)
    return {
        "tag": tag, "init_seed": init_seed, "local_iterations": local_iterations,
        "picard_iterations": picard_iterations, "steps_per_iter": steps_per_iter,
        "total_evals": total_evals, "total_time": total_time, "final_rel_rms": final_rel_rms,
        "history": history
    }

p2_results = {"variability": {}, "cost_comparison": {}}

# Part A: Training Variability across 5 independent initializations
print("\n" + "=" * 90)
print(" PART A: TRAINING VARIABILITY ACROSS INDEPENDENT INITIALISATIONS (DRIVER 0) ")
print(" Protocol: Frozen Causal3 (20 Picard iters x 1200 Adam steps, Width 128, FP64) ")
print("=" * 90, flush=True)

for s_init in [80000, 80001, 80002, 80003, 80004]:
    res = execute_run(f"causal3_init_{s_init}", init_seed=s_init, local_iterations=3, picard_iterations=20, steps_per_iter=1200)
    p2_results["variability"][s_init] = res
    with open("/content/benchmark/priority2/summary.json", "w") as f:
        json.dump(p2_results, f, indent=2)

var_errs = [r["final_rel_rms"] for r in p2_results["variability"].values()]
mean_v = float(np.mean(var_errs))
std_v = float(np.std(var_errs))
min_v = float(np.min(var_errs))
max_v = float(np.max(var_errs))

print("\n" + "=" * 90)
print(" PART A SUMMARY: TRAINING VARIABILITY (5 SEEDS) ")
print(f" Mean Rel RMS:    {mean_v:.4%}")
print(f" Std Dev:         {std_v:.4%}")
print(f" Spread [Min,Max]: [{min_v:.4%}, {max_v:.4%}]")
print(f" Coeff of Var:    {std_v / mean_v:.2%}")
print("=" * 90, flush=True)

# Part B: Fair Cost Comparisons
print("\n" + "=" * 90)
print(" PART B: FAIR COMPUTATIONAL EFFORT & SPEEDUP COMPARISON ")
print(" Comparing Causal3 vs Causal1 under matched iterations and matched evaluations ")
print("=" * 90, flush=True)

# Baseline Causal3 from init 80000
p2_results["cost_comparison"]["causal3_20iter_1200step"] = p2_results["variability"][80000]

# Arm 1: Causal1 Matched Iterations (20 iters x 1200 steps)
res_c1_matched_iter = execute_run("causal1_matched_iterations_20x1200", init_seed=80000, local_iterations=1, picard_iterations=20, steps_per_iter=1200)
p2_results["cost_comparison"]["causal1_matched_iterations"] = res_c1_matched_iter
with open("/content/benchmark/priority2/summary.json", "w") as f:
    json.dump(p2_results, f, indent=2)

# Arm 2: Causal1 Matched Evals & Matched Gradient Steps (60 iters x 400 steps)
res_c1_matched_effort = execute_run("causal1_matched_evals_and_steps_60x400", init_seed=80000, local_iterations=1, picard_iterations=60, steps_per_iter=400)
p2_results["cost_comparison"]["causal1_matched_effort"] = res_c1_matched_effort
with open("/content/benchmark/priority2/summary.json", "w") as f:
    json.dump(p2_results, f, indent=2)

print("\n" + "=" * 90)
print("                   PRIORITY 2: COMPLETE FINAL AUDIT                   ")
print("=" * 90)
print(f"Variability (5 seeds) -> Mean: {mean_v:.4%} | Std: {std_v:.4%} | Range: [{min_v:.4%}, {max_v:.4%}]")
print(f"Causal3 (20 iter, 15360 evals, 24000 steps):    {p2_results['cost_comparison']['causal3_20iter_1200step']['final_rel_rms']:.4%}")
print(f"Causal1 (20 iter,  5120 evals, 24000 steps):    {p2_results['cost_comparison']['causal1_matched_iterations']['final_rel_rms']:.4%}")
print(f"Causal1 (60 iter, 15360 evals, 24000 steps):    {p2_results['cost_comparison']['causal1_matched_effort']['final_rel_rms']:.4%}")
print("=" * 90, flush=True)


     PRIORITY 2: TRAINING VARIABILITY & FAIR COST COMPARISONS     
Device: NVIDIA A100-SXM4-80GB (85.1 GB VRAM)

 PART A: TRAINING VARIABILITY ACROSS INDEPENDENT INITIALISATIONS (DRIVER 0) 
 Protocol: Frozen Causal3 (20 Picard iters x 1200 Adam steps, Width 128, FP64) 

[START] causal3_init_80000
InitSeed=80000 | K_local=3 | Iters=20 | Steps/Iter=1200
[causal3_init_80000 | ITER  1/20] ( 16.4s, evals:   765) Best step: 1200 | Rel RMS: 10.3520%
[causal3_init_80000 | ITER  5/20] ( 78.2s, evals:  3825) Best step:  900 | Rel RMS: 7.0050%
[causal3_init_80000 | ITER 10/20] (156.8s, evals:  7650) Best step: 1200 | Rel RMS: 5.8357%
[causal3_init_80000 | ITER 15/20] (233.7s, evals: 11475) Best step:  900 | Rel RMS: 5.4369%
[causal3_init_80000 | ITER 20/20] (313.8s, evals: 15300) Best step:    0 | Rel RMS: 5.3318%
>>> causal3_init_80000 COMPLETE in 313.8s | Final Rel RMS: 5.3318% <<<


[START] causal3_init_80001
InitSeed=80001 | K_local=3 | Iters=20 | Steps/Iter=1200
[causal3_init_80001 | ITER  1